# RAFT-CrowdFlow from the beginning

This notebook contains the full starting implementation. It does **not** reproduce the rejected paper's tables. Defaults implement motion-only CFI; externally supplied pedestrian density is optional and is never inferred from blurred flow. No crowd-crush prediction or real-time performance is established.

Select **Runtime > Change runtime type > GPU** in Colab. Run cells in order. No original notebook or private manuscript is required. All embedded source files are checksum-pinned and regenerated from the local repository builder.


In [ ]:
from pathlib import Path
import hashlib, json, os, sys, subprocess

ROOT = Path.cwd() / 'raftcrowd_start'
ROOT.mkdir(exist_ok=True)
FILES = {'src/raftcrowd/__init__.py': {'sha256': '0fa6152b569df1fc84b12c4ad32d223f1ac08763a721119a9a59a72d2335d0dd', 'text': '"""Prospective RAFT-CrowdFlow reference implementation; not recovered paper results."""\n\n__version__ = "0.1.0"\n'}, 'src/raftcrowd/cli.py': {'sha256': 'eaf86076d9527f0a0cc9cfb13958466fffae0999b2c62bbcafa40d7ffd61424c', 'text': '"""Command-line workflow; importing help never imports PyTorch."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nfrom .io import sha256, write_json\n\n\ndef video_frames(video, output):\n    import cv2\n    video = Path(video).resolve()\n    original_hash = sha256(video)\n    capture = cv2.VideoCapture(str(video))\n    if not capture.isOpened():\n        raise ValueError(f"Cannot open video {video}")\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    fps = float(capture.get(cv2.CAP_PROP_FPS))\n    index, timestamps = 0, []\n    try:\n        while True:\n            ok, image = capture.read()\n            if not ok:\n                break\n            filename = output / f"frame_{index:06d}.png"\n            if not cv2.imwrite(str(filename), image):\n                raise RuntimeError(f"Could not write frame {filename}")\n            timestamps.append(float(capture.get(cv2.CAP_PROP_POS_MSEC)))\n            index += 1\n    finally:\n        capture.release()\n    if index < 2:\n        raise ValueError("Video yielded fewer than two frames")\n    if sha256(video) != original_hash:\n        raise ValueError("Video changed during decoding")\n    write_json(output / "decode.json", {"video": str(video), "sha256": original_hash,\n                                      "frames": index, "nominal_fps": fps,\n                                      "decoder_timestamps_ms_unverified": timestamps,\n                                      "note": "OpenCV timestamps are diagnostic; no native-PTS alignment claim"})\n    return index\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description="Prospective RAFT-CrowdFlow implementation; no recovered paper results")\n    sub = parser.add_subparsers(dest="command", required=True)\n    extract = sub.add_parser("extract", help="Frozen RAFT flow and numeric cues for all manifest rows")\n    extract.add_argument("--manifest", required=True)\n    extract.add_argument("--config", required=True)\n    extract.add_argument("--out", required=True)\n    fit = sub.add_parser("calibrate", help="Train-only scales and validation-only thresholds")\n    fit.add_argument("--cache", required=True)\n    fit.add_argument("--out", required=True)\n    evaluation = sub.add_parser("evaluate", help="Fixed-threshold test evaluation and all declared ablations")\n    evaluation.add_argument("--cache", required=True)\n    evaluation.add_argument("--model", required=True)\n    evaluation.add_argument("--out", required=True)\n    panels = sub.add_parser("visualize", help="Six panels from a single cached flow")\n    panels.add_argument("--cache", required=True)\n    panels.add_argument("--model", required=True)\n    panels.add_argument("--video-id", required=True)\n    panels.add_argument("--frame-index", required=True, type=int)\n    panels.add_argument("--out", required=True)\n    frames = sub.add_parser("video-frames", help="Extract every decoded video frame to a new directory")\n    frames.add_argument("--video", required=True)\n    frames.add_argument("--out", required=True)\n    args = parser.parse_args(argv)\n    if args.command == "video-frames":\n        print("Decoded frames:", video_frames(args.video, args.out))\n        return\n    from .pipeline import calibrate, evaluate_cache, prepare_cache\n    if args.command == "extract":\n        prepare_cache(args.manifest, args.config, args.out)\n    elif args.command == "calibrate":\n        model = calibrate(args.cache, args.out)\n        print(json.dumps({"thresholds": model["thresholds"], "normalization": model["normalization"]}, indent=2))\n    elif args.command == "evaluate":\n        report = evaluate_cache(args.cache, args.model, args.out)\n        print(json.dumps({k: v["pooled"] for k, v in report["methods"].items()}, indent=2))\n    else:\n        from .visualize import visualize\n        print(visualize(args.cache, args.model, args.video_id, args.frame_index, args.out))\n\n\nif __name__ == "__main__":\n    main()\n'}, 'src/raftcrowd/features.py': {'sha256': '8d7b28bb31d4beb99ef5ef1db023e447bac2eaa4e8dd67199932853e433956e1', 'text': '"""Numeric motion cues for a documented RAFT-CrowdFlow starting implementation.\n\nThese choices are prospective implementation choices, not a reconstruction of\nthe rejected paper\'s experiments. Flow is horizontal/vertical displacement in\npixels per observed frame pair. Derivatives use a unit image-pixel grid. The\noptional density input must be a separately obtained, aligned pedestrian-density\nmap; flow magnitude is never substituted for pedestrian density.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom numbers import Integral\n\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass CueConfig:\n    """Motion segmentation settings, fixed before held-out evaluation.\n\n    K-means clusters moving pixels by (u, v), without temporal label tracking.\n    A seeded bounded subsample fits the centers; every moving pixel is assigned.\n    Pixels with magnitude <= min_motion are background, labeled -1. This gate\n    deliberately excludes near-static regions and does not detect static crushes.\n    """\n\n    clusters: int = 4\n    min_motion: float = 0.5\n    kmeans_iterations: int = 20\n    kmeans_samples: int = 4096\n    seed: int = 7\n\n    def __post_init__(self) -> None:\n        for name in ("clusters", "kmeans_iterations", "kmeans_samples"):\n            value = getattr(self, name)\n            if isinstance(value, bool) or not isinstance(value, Integral) or value < 1:\n                raise ValueError(f"{name} must be a positive integer")\n        if not np.isfinite(self.min_motion) or self.min_motion < 0:\n            raise ValueError("min_motion must be finite and nonnegative")\n        if isinstance(self.seed, bool) or not isinstance(self.seed, Integral) or self.seed < 0:\n            raise ValueError("seed must be a nonnegative integer")\n\n\ndef _nearest(points: np.ndarray, centers: np.ndarray) -> np.ndarray:\n    """Bound temporary memory when assigning full-resolution frames."""\n    labels = np.empty(len(points), dtype=np.int32)\n    for start in range(0, len(points), 65536):\n        stop = min(start + 65536, len(points))\n        delta = points[start:stop, None, :] - centers[None, :, :]\n        labels[start:stop] = np.argmin(np.sum(delta * delta, axis=2), axis=1)\n    return labels\n\n\ndef _cluster(points: np.ndarray, config: CueConfig) -> np.ndarray:\n    rng = np.random.default_rng(config.seed)\n    if len(points) > config.kmeans_samples:\n        indices = np.sort(rng.choice(len(points), config.kmeans_samples, replace=False))\n        sample = points[indices]\n    else:\n        sample = points\n    unique = np.unique(sample, axis=0)\n    count = min(config.clusters, len(unique))\n    # A shared scale preserves Euclidean geometry in (u, v). Do not separately\n    # whiten the axes, which would change the relative directional distances.\n    scale = max(float(np.median(np.hypot(sample[:, 0], sample[:, 1]))), 1e-12)\n    sample = sample / scale\n    unique = unique / scale\n    centers = [unique[int(rng.integers(len(unique)))]]\n    nearest_distance = np.sum((unique - centers[0]) ** 2, axis=1)\n    for _ in range(1, count):\n        centers.append(unique[int(np.argmax(nearest_distance))])\n        nearest_distance = np.minimum(\n            nearest_distance, np.sum((unique - centers[-1]) ** 2, axis=1)\n        )\n    centers = np.asarray(centers)\n    for _ in range(config.kmeans_iterations):\n        assignments = _nearest(sample, centers)\n        updated = centers.copy()\n        for label in range(count):\n            selected = assignments == label\n            if np.any(selected):\n                updated[label] = np.mean(sample[selected], axis=0)\n        if np.allclose(updated, centers, rtol=0, atol=1e-8):\n            centers = updated\n            break\n        centers = updated\n    return _nearest(points / scale, centers)\n\n\ndef compute_cues(\n    flow: np.ndarray,\n    config: CueConfig,\n    density: np.ndarray | None = None,\n    segment: bool = True,\n) -> dict[str, np.ndarray]:\n    """Return numeric cue maps plus ``mask`` and integer ``segments``.\n\n    Input shape is (height, width, 2), with each spatial dimension >= 2.\n    Returned numeric maps are float64, finite, and zero outside the motion mask:\n    ``magnitude``, ``compression``, ``direction_variance``,\n    ``magnitude_variance``, and optionally ``pressure`` = density * magnitude.\n    Direction variance is circular variance 1 - |mean(exp(i theta))|, rather\n    than the paper\'s ambiguous ordinary variance of wrapped angles. Regional\n    magnitude variance is population variance (ddof=0). ``segment=False``\n    assigns all moving pixels to one region, as a no-segmentation control.\n\n    No input is modified. No image/color normalization is performed here.\n    A separate training-only fitted normalizer must precede score fusion.\n    """\n    if not isinstance(config, CueConfig):\n        raise TypeError("config must be a CueConfig")\n    field = np.asarray(flow, dtype=np.float64)\n    if field.ndim != 3 or field.shape[-1] != 2 or min(field.shape[:2]) < 2:\n        raise ValueError("flow must have shape (H, W, 2), with H and W >= 2")\n    if not np.all(np.isfinite(field)):\n        raise ValueError("flow must contain only finite values")\n    u, v = field[..., 0], field[..., 1]\n    magnitude = np.hypot(u, v)\n    if not np.all(np.isfinite(magnitude)):\n        raise ValueError("flow magnitude exceeds numeric range")\n    mask = magnitude > config.min_motion\n    labels = np.full(mask.shape, -1, dtype=np.int32)\n    if np.any(mask):\n        labels[mask] = _cluster(field[mask], config) if segment else 0\n    compression = np.maximum(\n        0.0, -(np.gradient(u, axis=1) + np.gradient(v, axis=0))\n    )\n    direction_variance = np.zeros(mask.shape, dtype=np.float64)\n    magnitude_variance = np.zeros(mask.shape, dtype=np.float64)\n    for label in np.unique(labels[mask]):\n        selected = labels == label\n        resultant = np.hypot(\n            np.mean(u[selected] / magnitude[selected]),\n            np.mean(v[selected] / magnitude[selected]),\n        )\n        direction_variance[selected] = np.clip(1.0 - resultant, 0.0, 1.0)\n        magnitude_variance[selected] = np.var(magnitude[selected], ddof=0)\n    result = {\n        "magnitude": np.where(mask, magnitude, 0.0),\n        "compression": np.where(mask, compression, 0.0),\n        "direction_variance": direction_variance,\n        "magnitude_variance": magnitude_variance,\n        "mask": mask,\n        "segments": labels,\n    }\n    if density is not None:\n        density_map = np.asarray(density, dtype=np.float64)\n        if density_map.shape != mask.shape:\n            raise ValueError("density must have exactly the flow spatial shape")\n        if not np.all(np.isfinite(density_map)) or np.any(density_map < 0):\n            raise ValueError("density must be finite and nonnegative")\n        result["pressure"] = np.where(mask, density_map * magnitude, 0.0)\n    if any(not np.all(np.isfinite(value)) for value in result.values()):\n        raise ValueError("cue computation exceeded numeric range")\n    return result\n\n\ndef aggregate_map(\n    values: np.ndarray,\n    top_fraction: float = 0.01,\n    mask: np.ndarray | None = None,\n) -> float:\n    """Mean of the largest ceil(top_fraction * N) selected pixel scores.\n\n    Without a mask N is the full image area, including zero-valued background;\n    this fixes the scored area across frames. Passing a mask changes N and must\n    be declared as a separate protocol. An empty selected mask returns 0.\n    """\n    array = np.asarray(values, dtype=np.float64)\n    if array.ndim != 2 or array.size == 0 or not np.all(np.isfinite(array)):\n        raise ValueError("values must be a nonempty finite two-dimensional map")\n    if np.any(array < 0):\n        raise ValueError("score map must be nonnegative")\n    if not np.isfinite(top_fraction) or not 0 < top_fraction <= 1:\n        raise ValueError("top_fraction must be in (0, 1]")\n    if mask is None:\n        selected = array.ravel()\n    else:\n        selected_mask = np.asarray(mask)\n        if selected_mask.shape != array.shape or selected_mask.dtype != np.bool_:\n            raise ValueError("mask must be Boolean and match the score map")\n        selected = array[selected_mask]\n    if selected.size == 0:\n        return 0.0\n    count = max(1, int(np.ceil(top_fraction * selected.size)))\n    return float(np.mean(np.partition(selected, selected.size - count)[-count:]))\n'}, 'src/raftcrowd/io.py': {'sha256': '8886087154c747dfcfc563a99b073dfd4fb427a234205227e4c362abd2cd43db', 'text': '"""Strict manifests, prospective configuration and provenance without GPU imports."""\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport platform\nimport re\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nDEFAULT_CONFIG = {\n    "seed": 7, "height": 360, "width": 640, "weights": "C_T_V1", "updates": 12,\n    "device": "cuda", "mixed_precision": False, "smooth_sigma": 1.0,\n    "clusters": 4, "min_motion": 0.5, "kmeans_iterations": 20, "kmeans_samples": 4096,\n    "density_mode": "none", "top_fraction": 0.01, "normalization_quantile": 0.99,\n    "normalization_samples_per_frame": 128, "normalization_max_samples": 50000,\n}\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef identity(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()\n\n\ndef write_json(path, value):\n    # All scientific outputs are exclusive-create. Failed stages leave partial outputs\n    # for inspection; a new path is required for retrying.\n    with Path(path).open("x", encoding="utf-8", newline="\\n") as handle:\n        json.dump(value, handle, indent=2, sort_keys=True, allow_nan=False)\n        handle.write("\\n")\n\n\ndef source_hashes():\n    root = Path(__file__).parent\n    return {p.name: sha256(p) for p in sorted(root.glob("*.py"))}\n\n\ndef environment():\n    from importlib.metadata import PackageNotFoundError, version\n    packages = {}\n    for name in ["numpy", "opencv-python", "opencv-python-headless", "torch", "torchvision"]:\n        try:\n            packages[name] = version(name)\n        except PackageNotFoundError:\n            pass\n    return {"python": sys.version, "platform": platform.platform(), "packages": packages,\n            "command": sys.argv, "source_sha256": source_hashes()}\n\n\ndef load_config(path=None):\n    config = dict(DEFAULT_CONFIG)\n    if path is not None:\n        supplied = json.loads(Path(path).read_text(encoding="utf-8"))\n        if not isinstance(supplied, dict) or supplied.keys() - config.keys():\n            raise ValueError("Configuration must be an object with known keys only")\n        config.update(supplied)\n    for name in ["seed", "height", "width", "updates", "clusters", "kmeans_iterations",\n                 "kmeans_samples", "normalization_samples_per_frame", "normalization_max_samples"]:\n        if type(config[name]) is not int or config[name] < (0 if name == "seed" else 1):\n            raise ValueError(f"Invalid integer configuration: {name}")\n    if min(config["height"], config["width"]) < 128:\n        raise ValueError("height/width must be >=128 for RAFT-large")\n    for name in ["smooth_sigma", "min_motion", "top_fraction", "normalization_quantile"]:\n        if isinstance(config[name], bool) or not isinstance(config[name], (int, float)) or not np.isfinite(config[name]):\n            raise ValueError(f"Invalid finite configuration: {name}")\n    if config["smooth_sigma"] < 0 or config["min_motion"] < 0:\n        raise ValueError("Smoothing/motion gate must be nonnegative")\n    if not 0 < config["top_fraction"] <= 1 or not 0 < config["normalization_quantile"] <= 1:\n        raise ValueError("Quantile and top fraction must be in (0,1]")\n    if not isinstance(config["weights"], str) or config["weights"] not in {"C_T_V1", "C_T_V2", "C_T_SKHT_V1", "C_T_SKHT_V2"}:\n        raise ValueError("Use an explicit supported RAFT-large weight enum, not DEFAULT")\n    if not isinstance(config["device"], str) or config["device"] not in {"cuda", "cpu"} or type(config["mixed_precision"]) is not bool:\n        raise ValueError("device must be cuda/cpu; mixed_precision must be boolean")\n    if config["mixed_precision"] and config["device"] != "cuda":\n        raise ValueError("Mixed precision requires CUDA")\n    if not isinstance(config["density_mode"], str) or config["density_mode"] not in {"none", "external"}:\n        raise ValueError("density_mode must be none or external")\n    return config\n\n\ndef load_manifest(path, config):\n    path = Path(path).resolve()\n    with path.open(newline="", encoding="utf-8-sig") as handle:\n        reader = csv.DictReader(handle)\n        if not reader.fieldnames or len(reader.fieldnames) != 6 or set(reader.fieldnames) != {"id", "split", "group", "frames", "labels", "density"}:\n            raise ValueError("CSV columns must be id,split,group,frames,labels,density")\n        rows = list(reader)\n    if not rows:\n        raise ValueError("Manifest is empty")\n    ids, directories, groups = set(), set(), {}\n    records = []\n    for row in rows:\n        if any(value is None for value in row.values()) or None in row:\n            raise ValueError("Malformed CSV row")\n        row = {k: v.strip() for k, v in row.items()}\n        if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", row["id"]) or row["id"] in ids:\n            raise ValueError("Video ids must be unique safe names")\n        ids.add(row["id"])\n        if row["split"] not in {"train", "val", "test"} or not row["group"]:\n            raise ValueError("Each row requires train/val/test and a source group")\n        if groups.setdefault(row["group"], row["split"]) != row["split"]:\n            raise ValueError("Source group crosses splits")\n        if not row["frames"]:\n            raise ValueError("Frame directory is required")\n        directory = (path.parent / row["frames"]).resolve()\n        if directory in directories:\n            raise ValueError("Frame directory reused across rows")\n        directories.add(directory)\n        if not directory.is_dir():\n            raise FileNotFoundError(f"Missing frame directory: {directory}")\n        frames = sorted(p for p in directory.iterdir() if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg"})\n        if len(frames) < 2 or len({p.stem for p in frames}) != len(frames):\n            raise ValueError("Need >=2 images with unique stems, sorted in chronological filename order")\n        label_path = (path.parent / row["labels"]).resolve() if row["labels"] else None\n        if label_path is None:\n            if row["split"] != "train":\n                raise ValueError("Validation/test rows need per-frame binary .npy labels")\n            labels = np.zeros(len(frames), dtype=np.uint8)\n        else:\n            labels = np.load(label_path, allow_pickle=False)\n        if labels.shape != (len(frames),) or not np.isin(labels, [0, 1]).all():\n            raise ValueError("Labels must be a binary vector with one entry per input frame")\n        if row["split"] == "train" and np.any(labels):\n            raise ValueError("Calibration training data must be normal-only")\n        density = (path.parent / row["density"]).resolve() if row["density"] else None\n        if config["density_mode"] == "external" and (density is None or not density.is_dir()):\n            raise ValueError("External density mode requires a density directory for every video")\n        if config["density_mode"] == "none" and density is not None:\n            raise ValueError("Density paths supplied but density_mode=none; resolve configuration explicitly")\n        records.append({"id": row["id"], "split": row["split"], "group": row["group"],\n                        "frames": frames, "labels": labels.astype(np.uint8),\n                        "label_path": label_path, "density": density})\n    if {r["split"] for r in records} != {"train", "val", "test"}:\n        raise ValueError("Manifest must include train, val and test")\n    return records\n'}, 'src/raftcrowd/metrics.py': {'sha256': 'f74a65113d233bed9ee38be76ea9b053d186f43edcbf59f2d68d6caf251d25b0', 'text': '"""Frame-level binary metrics with explicit, validation-only threshold selection.\n\nHigher scores mean more anomalous. No test-set threshold optimization is performed\nby :func:`evaluate`; its caller must supply a previously frozen threshold.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef _validated(labels, scores, *, require_both: bool = False):\n    y = np.asarray(labels)\n    s = np.asarray(scores)\n    if y.ndim != 1 or s.ndim != 1 or y.shape != s.shape or not y.size:\n        raise ValueError("labels and scores must be matching nonempty 1-D arrays")\n    if y.dtype.kind not in "biuf" or s.dtype.kind not in "biuf":\n        raise ValueError("labels and scores must be real numeric arrays")\n    if not np.all(np.isfinite(y)) or not np.all((y == 0) | (y == 1)):\n        raise ValueError("labels must contain only binary 0/1 values")\n    s = s.astype(np.float64, copy=False)\n    if not np.all(np.isfinite(s)):\n        raise ValueError("scores must be finite")\n    y = y.astype(np.int64, copy=False)\n    if require_both and (not np.any(y == 0) or not np.any(y == 1)):\n        raise ValueError("both normal (0) and anomalous (1) labels are required")\n    return y, s\n\n\ndef _counts_at_distinct_scores(y: np.ndarray, s: np.ndarray):\n    # Keeping tied scores together makes ROC/AUC independent of input row order.\n    order = np.argsort(-s, kind="stable")\n    ranked = s[order]\n    ends = np.r_[np.flatnonzero(np.diff(ranked)), len(ranked) - 1]\n    tp = np.cumsum(y[order], dtype=np.int64)[ends]\n    fp = ends + 1 - tp\n    return tp, fp, ranked[ends]\n\n\ndef roc_curve(labels, scores):\n    """Return ``(fpr, tpr, thresholds)`` with ties grouped exactly.\n\n    The initial threshold is +inf, representing no positive predictions. This\n    sentinel is for the curve only, not an admissible operational threshold.\n    Both label classes must be present.\n    """\n    y, s = _validated(labels, scores, require_both=True)\n    tp, fp, thresholds = _counts_at_distinct_scores(y, s)\n    tpr = np.r_[0.0, tp / np.count_nonzero(y == 1)]\n    fpr = np.r_[0.0, fp / np.count_nonzero(y == 0)]\n    return fpr, tpr, np.r_[np.inf, thresholds]\n\n\ndef roc_auc(labels, scores) -> float:\n    """Trapezoidal ROC AUC; a tied positive/negative pair contributes 0.5."""\n    fpr, tpr, _ = roc_curve(labels, scores)\n    return float(np.sum(np.diff(fpr) * (tpr[:-1] + tpr[1:]) * 0.5))\n\n\ndef eer(labels, scores) -> float:\n    """Equal-error rate from linear interpolation of the empirical ROC.\n\n    Interpolates between neighboring curve points where FPR and FNR cross.\n    This is a descriptive curve statistic, not a deployed alarm threshold.\n    """\n    fpr, tpr, _ = roc_curve(labels, scores)\n    fnr = 1.0 - tpr\n    difference = fpr - fnr\n    index = int(np.flatnonzero(difference >= 0)[0])\n    if difference[index] == 0:\n        return float(fpr[index])\n    left = index - 1\n    fraction = -difference[left] / (difference[index] - difference[left])\n    return float(fpr[left] + fraction * (fpr[index] - fpr[left]))\n\n\ndef fixed_metrics(labels, scores, threshold) -> dict:\n    """Precision/recall/F1 and counts at ``score >= threshold``.\n\n    Zero-denominator precision, recall and F1 are defined to be zero. Single-class\n    labels are allowed here, unlike for ROC AUC and EER.\n    """\n    y, s = _validated(labels, scores)\n    value = np.asarray(threshold)\n    if value.ndim != 0 or value.dtype.kind not in "biuf" or not np.isfinite(value):\n        raise ValueError("threshold must be a finite real scalar")\n    predicted = s >= float(value)\n    positive = y == 1\n    tp = int(np.count_nonzero(predicted & positive))\n    fp = int(np.count_nonzero(predicted & ~positive))\n    tn = int(np.count_nonzero(~predicted & ~positive))\n    fn = int(np.count_nonzero(~predicted & positive))\n    precision = tp / (tp + fp) if tp + fp else 0.0\n    recall = tp / (tp + fn) if tp + fn else 0.0\n    denominator = 2 * tp + fp + fn\n    f1 = 2 * tp / denominator if denominator else 0.0\n    return {\n        "precision": precision,\n        "recall": recall,\n        "f1": f1,\n        "tp": tp,\n        "fp": fp,\n        "tn": tn,\n        "fn": fn,\n    }\n\n\ndef select_threshold(labels, scores) -> float:\n    """Maximize validation F1, choosing the highest score threshold on ties.\n\n    Call only with the designated validation set. Both classes are required.\n    The function cannot determine a split\'s provenance: callers must enforce it.\n    Candidate thresholds are the distinct observed scores. With at least one\n    positive label, a threshold predicting nothing cannot improve maximum F1.\n    """\n    y, s = _validated(labels, scores, require_both=True)\n    tp, fp, thresholds = _counts_at_distinct_scores(y, s)\n    fn = int(np.count_nonzero(y == 1)) - tp\n    f1 = 2.0 * tp / (2 * tp + fp + fn)\n    # thresholds are descending; argmax\'s first occurrence resolves ties high.\n    return float(thresholds[int(np.argmax(f1))])\n\n\ndef evaluate(labels, scores, threshold) -> dict:\n    """Evaluate one previously chosen threshold plus ROC AUC and EER."""\n    result = fixed_metrics(labels, scores, threshold)\n    result.update(roc_auc=roc_auc(labels, scores), eer=eer(labels, scores))\n    return result\n'}, 'src/raftcrowd/pipeline.py': {'sha256': 'c7e17d7f085c9a8a2d940184df262316e3268cab8b910bbebb7ccc9151262fdf', 'text': '"""Extract once, calibrate on train/validation, evaluate fixed methods on test."""\n\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .features import CueConfig, aggregate_map, compute_cues\nfrom .io import environment, identity, load_config, load_manifest, sha256, source_hashes, write_json\nfrom .metrics import evaluate, fixed_metrics, select_threshold\n\n\ndef cue_config(config):\n    return CueConfig(**{k: config[k] for k in ["clusters", "min_motion", "kmeans_iterations", "kmeans_samples", "seed"]})\n\n\ndef methods(config):\n    cues = ["compression", "direction_variance", "magnitude_variance"]\n    if config["density_mode"] == "external":\n        cues.append("pressure")\n    result = {"full": {"cues": cues, "segmented": True},\n              "magnitude_only": {"cues": ["magnitude"], "segmented": True},\n              "no_segmentation": {"cues": cues, "segmented": False}}\n    for cue in cues:\n        result[f"{cue}_only"] = {"cues": [cue], "segmented": True}\n        result[f"without_{cue}"] = {"cues": [c for c in cues if c != cue], "segmented": True}\n    return result\n\n\ndef read_rgb(path, config):\n    import cv2\n    image = cv2.imread(str(path), cv2.IMREAD_COLOR)\n    if image is None:\n        raise ValueError(f"Cannot decode frame {path}")\n    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)\n    return cv2.resize(image, (config["width"], config["height"]), interpolation=cv2.INTER_LINEAR)\n\n\ndef prepare_cache(manifest_path, config_path, output, estimator=None):\n    """An injected estimator is only for synthetic tests, and is marked in receipts."""\n    config = load_config(config_path)\n    records = load_manifest(manifest_path, config)\n    # Hash all originals before inference. Whole-video copies cannot masquerade as\n    # different splits. Group correctness still requires a human dataset audit.\n    seen_videos = {}\n    for record in records:\n        record["frame_sha256"] = [sha256(p) for p in record["frames"]]\n        video_identity = identity(record["frame_sha256"])\n        if video_identity in seen_videos:\n            raise ValueError("Identical frame sequence appears in multiple manifest rows")\n        seen_videos[video_identity] = record["id"]\n        record["density_sha256"] = []\n        if record["density"] is not None:\n            # Cheap coverage/shape audit before constructing a GPU model.\n            for frame in record["frames"][:-1]:\n                density_path = record["density"] / (frame.stem + ".npy")\n                density = np.load(density_path, allow_pickle=False, mmap_mode="r")\n                if density.shape != (config["height"], config["width"]) or not np.isfinite(density).all() or np.any(density < 0):\n                    raise ValueError("External density must be finite, nonnegative and aligned to the resized source-frame grid")\n                record["density_sha256"].append(sha256(density_path))\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    if estimator is None:\n        from .raft import RaftEstimator\n        estimator = RaftEstimator(config)\n    backend = getattr(estimator, "metadata", {"backend": "injected_unverified_test_estimator"})\n    meta = {"format": 1, "config": config, "manifest_sha256": sha256(manifest_path),\n            "environment": environment(), "estimator": backend, "videos": []}\n    import cv2\n    for record in records:\n        video_dir = output / record["id"]\n        video_dir.mkdir()\n        np.save(video_dir / "labels.npy", record["labels"], allow_pickle=False)\n        video = {"id": record["id"], "split": record["split"], "group": record["group"],\n                 "frames": [str(p) for p in record["frames"]],\n                 "frame_sha256": record["frame_sha256"],\n                 "label_source_sha256": sha256(record["label_path"]) if record["label_path"] else None,\n                 "labels_sha256": sha256(video_dir / "labels.npy"), "pairs": []}\n        first = read_rgb(record["frames"][0], config)\n        for index, frame in enumerate(record["frames"][1:], 1):\n            # Recheck the input hash as it is read; reject concurrent source changes.\n            if sha256(record["frames"][index - 1]) != record["frame_sha256"][index - 1] or sha256(frame) != record["frame_sha256"][index]:\n                raise ValueError("Input frame changed during extraction")\n            start = time.perf_counter()\n            second = read_rgb(frame, config)\n            flow, inference_seconds = estimator(first, second)\n            flow = np.asarray(flow, dtype=np.float32)\n            if flow.shape != (config["height"], config["width"], 2):\n                raise ValueError("Estimator output must match the unpadded resized image dimensions")\n            if config["smooth_sigma"] > 0:\n                flow = cv2.GaussianBlur(flow, (0, 0), config["smooth_sigma"])\n            density, density_hash = None, None\n            if record["density"] is not None:\n                # Optical flow is defined on the first image\'s pixel coordinates.\n                density_path = record["density"] / (record["frames"][index - 1].stem + ".npy")\n                density_hash = sha256(density_path)\n                if density_hash != record["density_sha256"][index - 1]:\n                    raise ValueError("External density changed during extraction")\n                density = np.load(density_path, allow_pickle=False)\n                if density.shape != flow.shape[:2]:\n                    raise ValueError("External density must be aligned to the resized source-frame grid; automatic resizing is forbidden")\n            cues = compute_cues(flow, cue_config(config), density=density)\n            global_cues = compute_cues(flow, cue_config(config), density=density, segment=False)\n            arrays = {name: value.astype(np.float32) for name, value in cues.items()\n                      if name not in {"mask", "segments"}}\n            arrays.update(mask=cues["mask"], segments=cues["segments"].astype(np.int32), flow=flow,\n                          global_direction_variance=global_cues["direction_variance"].astype(np.float32),\n                          global_magnitude_variance=global_cues["magnitude_variance"].astype(np.float32))\n            if any(not np.isfinite(value).all() for value in arrays.values()):\n                raise ValueError("Cached cue/flow exceeds finite float32 range")\n            pair_name = f"pair_{index:06d}.npz"\n            np.savez_compressed(video_dir / pair_name, **arrays)\n            video["pairs"].append({"index": index, "file": pair_name,\n                                   "sha256": sha256(video_dir / pair_name),\n                                   "source_density_sha256": density_hash,\n                                   "inference_seconds": float(inference_seconds),\n                                   "extraction_seconds": time.perf_counter() - start})\n            first = second\n        meta["videos"].append(video)\n        print(f"Extracted {record[\'id\']}: {len(video[\'pairs\'])} pairs", flush=True)\n    # Receipt last. A partial directory cannot be calibrated or evaluated.\n    write_json(output / "cache.json", meta)\n    write_json(output / "complete.json", {"cache_sha256": sha256(output / "cache.json")})\n    return meta\n\n\ndef load_cache(root):\n    root = Path(root)\n    receipt = json.loads((root / "complete.json").read_text(encoding="utf-8"))\n    if sha256(root / "cache.json") != receipt["cache_sha256"]:\n        raise ValueError("Cache metadata checksum mismatch")\n    meta = json.loads((root / "cache.json").read_text(encoding="utf-8"))\n    if meta["environment"]["source_sha256"] != source_hashes():\n        raise ValueError("Implementation changed since extraction; preserve old code or create a new cache")\n    return root, meta, receipt["cache_sha256"]\n\n\ndef iter_pairs(root, meta, split=None):\n    for video in meta["videos"]:\n        if split is not None and video["split"] != split:\n            continue\n        label_path = root / video["id"] / "labels.npy"\n        if sha256(label_path) != video["labels_sha256"]:\n            raise ValueError("Cached labels changed")\n        labels = np.load(label_path, allow_pickle=False)\n        for pair in video["pairs"]:\n            file = root / video["id"] / pair["file"]\n            if sha256(file) != pair["sha256"]:\n                raise ValueError(f"Cached pair checksum mismatch: {file}")\n            with np.load(file, allow_pickle=False) as shard:\n                arrays = {key: shard[key] for key in shard.files}\n            yield video, pair, int(labels[pair["index"]]), arrays\n\n\ndef score_map(arrays, method, scales):\n    parts = []\n    for cue in method["cues"]:\n        key = "global_" + cue if not method["segmented"] and cue in {"direction_variance", "magnitude_variance"} else cue\n        values = np.asarray(arrays[key], dtype=np.float64)\n        parts.append(np.clip(values / scales[key], 0, 1))\n    return np.mean(parts, axis=0)  # equal weights, sum=1; never fitted on test\n\n\ndef score_split(root, meta, model, split):\n    rows = []\n    for video, pair, label, arrays in iter_pairs(root, meta, split):\n        row = {"video": video["id"], "index": pair["index"], "label": label}\n        for name, method in model["methods"].items():\n            row[name] = aggregate_map(score_map(arrays, method, model["scales"]), meta["config"]["top_fraction"])\n        rows.append(row)\n    return rows\n\n\ndef calibrate(cache, output):\n    root, meta, cache_hash = load_cache(cache)\n    config = meta["config"]\n    definitions = methods(config)\n    cues = sorted({("global_" + cue if not method["segmented"] and cue in\n                    {"direction_variance", "magnitude_variance"} else cue)\n                   for method in definitions.values() for cue in method["cues"]})\n    # Bounded random-priority reservoirs are deterministic for a fixed manifest\n    # traversal order. Only normal training frames enter these reservoirs.\n    rng = np.random.default_rng(config["seed"])\n    samples = {cue: np.empty(0) for cue in cues}\n    priorities = {cue: np.empty(0) for cue in cues}\n    training_pairs = 0\n    for _, _, label, arrays in iter_pairs(root, meta, "train"):\n        if label != 0:\n            raise ValueError("Anomalous training label in cache")\n        training_pairs += 1\n        active = np.flatnonzero(arrays["mask"])\n        if not len(active):\n            continue\n        indices = rng.choice(active, min(len(active), config["normalization_samples_per_frame"]), replace=False)\n        for cue in cues:\n            samples[cue] = np.concatenate((samples[cue], arrays[cue].ravel()[indices]))\n            priorities[cue] = np.concatenate((priorities[cue], rng.random(len(indices))))\n            cap = config["normalization_max_samples"]\n            if len(samples[cue]) > cap:\n                keep = np.argpartition(priorities[cue], -cap)[-cap:]\n                samples[cue], priorities[cue] = samples[cue][keep], priorities[cue][keep]\n    if not training_pairs:\n        raise ValueError("No training pairs")\n    scales = {}\n    zero_scale_cues = []\n    for cue in cues:\n        value = float(np.quantile(samples[cue], config["normalization_quantile"])) if len(samples[cue]) else 0.0\n        if value <= 1e-12:\n            zero_scale_cues.append(cue)\n            value = 1.0  # documented fallback when training cue has no positive range\n        scales[cue] = value\n    model = {"format": 1, "cache_sha256": cache_hash, "methods": definitions, "scales": scales,\n             "normalization": {"training_pairs": training_pairs, "quantile": config["normalization_quantile"],\n                               "sample_counts": {k: len(v) for k, v in samples.items()},\n                               "zero_scale_fallback_cues": zero_scale_cues},\n             "threshold_source": "validation_labels_only", "thresholds": {}, "environment": environment()}\n    validation = score_split(root, meta, model, "val")\n    labels = np.array([r["label"] for r in validation])\n    for name in definitions:\n        model["thresholds"][name] = select_threshold(labels, np.array([r[name] for r in validation]))\n    model["validation_predictions"] = validation\n    Path(output).parent.mkdir(parents=True, exist_ok=True)\n    write_json(output, model)\n    return model\n\n\ndef load_model(path, cache_hash, meta):\n    model = json.loads(Path(path).read_text(encoding="utf-8"))\n    if model["cache_sha256"] != cache_hash:\n        raise ValueError("Calibrated model belongs to a different cache")\n    if model["methods"] != methods(meta["config"]):\n        raise ValueError("Method definitions changed")\n    if any(not math.isfinite(v) or v <= 0 for v in model["scales"].values()):\n        raise ValueError("Invalid normalization scales")\n    if set(model["thresholds"]) != set(model["methods"]) or not all(math.isfinite(v) for v in model["thresholds"].values()):\n        raise ValueError("Invalid frozen thresholds")\n    return model\n\n\ndef evaluate_cache(cache, model_path, output):\n    root, meta, cache_hash = load_cache(cache)\n    model = load_model(model_path, cache_hash, meta)\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    rows = score_split(root, meta, model, "test")\n    labels = np.array([r["label"] for r in rows])\n    report = {"status": "computed_from_supplied_inputs_not_historical_paper_results",\n              "cache_sha256": cache_hash, "model_sha256": sha256(model_path), "aggregation": "pooled_scored_frames",\n              "frame_alignment": "pair(t,t+1) predicts anomaly label[t+1]; first frame excluded",\n              "environment": environment(), "methods": {}}\n    for name in model["methods"]:\n        scores = np.array([r[name] for r in rows])\n        metrics = evaluate(labels, scores, model["thresholds"][name])\n        per_video = {}\n        for video_id in sorted({r["video"] for r in rows}):\n            chosen = [r for r in rows if r["video"] == video_id]\n            y = np.array([r["label"] for r in chosen])\n            s = np.array([r[name] for r in chosen])\n            if len(np.unique(y)) == 2:\n                per_video[video_id] = evaluate(y, s, model["thresholds"][name])\n            else:\n                per_video[video_id] = dict(fixed_metrics(y, s, model["thresholds"][name]),\n                                          roc_auc=None, eer=None, reason="single_class_video")\n        report["methods"][name] = {"threshold": model["thresholds"][name], "pooled": metrics, "per_video": per_video}\n    with (output / "frame_scores.csv").open("x", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=["video", "index", "label", *model["methods"]])\n        writer.writeheader()\n        writer.writerows(rows)\n    write_json(output / "metrics.json", report)\n    write_json(output / "complete.json", {"metrics_sha256": sha256(output / "metrics.json"),\n                                         "frame_scores_sha256": sha256(output / "frame_scores.csv")})\n    return report\n'}, 'src/raftcrowd/raft.py': {'sha256': 'd054953082bda9825c9870628921bf2dfd9ab8b84fa59bf8ae88aa62de8f0358', 'text': '"""Lazy Torchvision RAFT adapter: RGB, explicit weights, pad then crop, one flow."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport time\nimport numpy as np\n\n\nclass RaftEstimator:\n    def __init__(self, config):\n        # Import only during actual inference; feature/metric tests need no GPU stack.\n        import torch\n        import torchvision\n        from torchvision.models.optical_flow import Raft_Large_Weights, raft_large\n\n        self.torch = torch\n        self.config = config\n        self.device = torch.device(config["device"])\n        if self.device.type == "cuda" and not torch.cuda.is_available():\n            raise RuntimeError("CUDA unavailable. Select a GPU runtime or explicitly set device=cpu.")\n        if config["mixed_precision"] and self.device.type != "cuda":\n            raise ValueError("mixed_precision requires CUDA in this implementation")\n        torch.manual_seed(config["seed"])\n        if self.device.type == "cuda":\n            torch.cuda.manual_seed_all(config["seed"])\n        torch.backends.cudnn.benchmark = False\n        torch.backends.cudnn.deterministic = True\n        weights = Raft_Large_Weights[config["weights"]]\n        self.transform = weights.transforms()\n        self.model = raft_large(weights=weights, progress=True).to(self.device).eval()\n        parameter_hash = hashlib.sha256()\n        for name, tensor in sorted(self.model.state_dict().items()):\n            array = tensor.detach().cpu().contiguous().numpy()\n            parameter_hash.update(name.encode())\n            parameter_hash.update(str(array.dtype).encode())\n            parameter_hash.update(str(array.shape).encode())\n            parameter_hash.update(array.tobytes())\n        self.metadata = {\n            "backend": "torchvision.raft_large", "weights": config["weights"],\n            "weights_url": weights.url, "torch": torch.__version__,\n            "state_dict_sha256": parameter_hash.hexdigest(),\n            "torchvision": torchvision.__version__, "cuda_runtime": torch.version.cuda,\n            "device": str(self.device),\n            "hardware": torch.cuda.get_device_name(self.device) if self.device.type == "cuda" else "CPU",\n            "precision": "autocast_fp16" if config["mixed_precision"] else "fp32",\n            "updates": config["updates"], "batch_size": 1,\n            "timing_scope": "tensor upload + normalization + padding + RAFT + crop + CPU transfer; excludes decoding, resizing and CFI",\n            "warmup": "none; per-pair timings include cold start and are diagnostic, not a deployment benchmark",\n        }\n\n    def __call__(self, first_rgb, second_rgb):\n        torch = self.torch\n        from torch.nn import functional as F\n\n        if first_rgb.shape != second_rgb.shape or first_rgb.ndim != 3 or first_rgb.shape[2] != 3:\n            raise ValueError("RAFT requires two equally sized HxWx3 RGB arrays")\n        if first_rgb.dtype != np.uint8 or second_rgb.dtype != np.uint8:\n            raise ValueError("RGB images must be uint8")\n        h, w = first_rgb.shape[:2]\n        if min(h, w) < 128:\n            raise ValueError("RAFT-large requires image dimensions >=128")\n        if self.device.type == "cuda":\n            torch.cuda.synchronize(self.device)\n        start = time.perf_counter()\n        a = torch.from_numpy(np.ascontiguousarray(first_rgb)).permute(2, 0, 1)[None].to(self.device)\n        b = torch.from_numpy(np.ascontiguousarray(second_rgb)).permute(2, 0, 1)[None].to(self.device)\n        a, b = self.transform(a, b)\n        # Right/bottom padding preserves the original pixel origin and vector units.\n        pad = (0, (-w) % 8, 0, (-h) % 8)\n        a, b = F.pad(a, pad, mode="replicate"), F.pad(b, pad, mode="replicate")\n        with torch.inference_mode(), torch.autocast(\n            device_type=self.device.type, dtype=torch.float16,\n            enabled=self.config["mixed_precision"],\n        ):\n            predictions = self.model(a, b, num_flow_updates=self.config["updates"])\n            flow = predictions[-1][0, :, :h, :w].float().permute(1, 2, 0).cpu().numpy()\n        if self.device.type == "cuda":\n            torch.cuda.synchronize(self.device)\n        elapsed = time.perf_counter() - start\n        if not np.isfinite(flow).all():\n            raise ValueError("RAFT produced nonfinite flow")\n        return flow, elapsed\n'}, 'src/raftcrowd/visualize.py': {'sha256': 'a484f135e8d194eca9b9c15dd5329dbc6af79a3f194412b63bb2a8d30505754e', 'text': '"""Six panels derived from one checked cached flow, never relabeled as safety truth."""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .io import sha256\nfrom .pipeline import load_cache, load_model, read_rgb, score_map\n\n\ndef visualize(cache, model_path, video_id, frame_index, output):\n    import cv2\n    root, meta, cache_hash = load_cache(cache)\n    model = load_model(model_path, cache_hash, meta)\n    video = next((v for v in meta["videos"] if v["id"] == video_id), None)\n    if video is None:\n        raise ValueError("Unknown video id")\n    pair = next((p for p in video["pairs"] if p["index"] == frame_index), None)\n    if pair is None:\n        raise ValueError("frame-index must be a scored second-frame index, from 1 to N-1")\n    source = video["frames"][frame_index - 1]\n    if sha256(source) != video["frame_sha256"][frame_index - 1]:\n        raise ValueError("Source image identity changed")\n    file = root / video_id / pair["file"]\n    if sha256(file) != pair["sha256"]:\n        raise ValueError("Cached pair changed")\n    with np.load(file, allow_pickle=False) as shard:\n        arrays = {key: shard[key] for key in shard.files}\n    flow = arrays["flow"]\n    mag = np.hypot(flow[..., 0], flow[..., 1])\n    hsv = np.zeros((*flow.shape[:2], 3), np.uint8)\n    hsv[..., 0] = ((np.arctan2(flow[..., 1], flow[..., 0]) + np.pi) * 180 / (2 * np.pi)).astype(np.uint8)\n    hsv[..., 1] = 255\n    hsv[..., 2] = (255 * np.clip(mag / model["scales"]["magnitude"], 0, 1)).astype(np.uint8)\n    flow_color = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)\n\n    def heat(values):\n        return cv2.applyColorMap((255 * np.clip(values, 0, 1)).astype(np.uint8), cv2.COLORMAP_TURBO)\n\n    full = score_map(arrays, model["methods"]["full"], model["scales"])\n    segments = arrays["segments"]\n    palette = np.array([[255, 180, 30], [20, 210, 240], [190, 50, 220], [70, 220, 90], [80, 80, 250]], np.uint8)\n    grouped = np.zeros((*segments.shape, 3), np.uint8)\n    active = segments >= 0\n    grouped[active] = palette[segments[active] % len(palette)]\n    panels = [cv2.cvtColor(read_rgb(source, meta["config"]), cv2.COLOR_RGB2BGR), flow_color,\n              heat(full), heat(arrays["magnitude"] / model["scales"]["magnitude"]),\n              heat(arrays["compression"] / model["scales"]["compression"]), grouped]\n    titles = ["Source RGB (t)", "Flow direction / magnitude", "CFI motion score [0,1]",\n              "Magnitude / training scale", "Compression / training scale", "Motion groups (not people)"]\n    labeled = []\n    for panel, title in zip(panels, titles):\n        bar = np.zeros((32, panel.shape[1], 3), np.uint8)\n        text_width = cv2.getTextSize(title, cv2.FONT_HERSHEY_SIMPLEX, 1, 1)[0][0]\n        font_scale = min(0.5, (panel.shape[1] - 12) / max(text_width, 1))\n        cv2.putText(bar, title, (6, 22), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), 1, cv2.LINE_AA)\n        labeled.append(np.vstack((bar, panel)))\n    mosaic = np.vstack([np.hstack(labeled[i:i + 2]) for i in [0, 2, 4]])\n    output = Path(output)\n    if output.suffix.lower() != ".png":\n        raise ValueError("Visualization output must be a PNG path")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    ok, encoded = cv2.imencode(".png", mosaic)\n    if not ok:\n        raise RuntimeError("PNG encoding failed")\n    with output.open("xb") as handle:\n        handle.write(encoded.tobytes())\n    return output\n'}, 'tests/test_raftcrowd_features.py': {'sha256': '58fc4b56361aba84fb72d1eec6b15a0decf116b36c8613b4cd867ea7b3d3bfbb', 'text': 'import numpy as np\nimport pytest\n\nfrom raftcrowd.features import CueConfig, aggregate_map, compute_cues\n\n\ndef test_constant_translation_has_no_instability():\n    flow = np.zeros((8, 9, 2))\n    flow[..., 0] = 3.0\n    original = flow.copy()\n    cues = compute_cues(flow, CueConfig())\n    np.testing.assert_array_equal(flow, original)\n    np.testing.assert_allclose(cues["magnitude"], 3.0)\n    for name in ("compression", "direction_variance", "magnitude_variance"):\n        np.testing.assert_allclose(cues[name], 0.0, atol=1e-12)\n    assert "pressure" not in cues\n    assert np.all(cues["mask"])\n\n\ndef test_linear_convergence_has_correct_compression_and_expansion_does_not():\n    y, x = np.mgrid[-3:4, -4:5]\n    flow = np.stack((-0.5 * x, -0.25 * y), axis=-1)\n    cues = compute_cues(flow, CueConfig(min_motion=0.0))\n    np.testing.assert_allclose(cues["compression"][cues["mask"]], 0.75)\n    assert cues["compression"][3, 4] == 0.0  # Explicit moving-only policy.\n    expansion = compute_cues(-flow, CueConfig(min_motion=0.0))\n    np.testing.assert_allclose(expansion["compression"], 0.0)\n\n\ndef test_circular_variance_respects_angle_wraparound():\n    angles = np.array([[np.pi - 0.01, -np.pi + 0.01]] * 2)\n    flow = np.stack((np.cos(angles), np.sin(angles)), axis=-1)\n    cues = compute_cues(flow, CueConfig(clusters=1))\n    np.testing.assert_allclose(cues["direction_variance"], 1 - np.cos(0.01), atol=1e-12)\n    np.testing.assert_allclose(cues["magnitude_variance"], 0.0, atol=1e-12)\n\n\ndef test_population_magnitude_variance_and_no_segmentation_control():\n    flow = np.zeros((2, 2, 2))\n    flow[..., 0] = [[1.0, 3.0], [1.0, 3.0]]\n    config = CueConfig(clusters=2)\n    segmented = compute_cues(flow, config)\n    unsegmented = compute_cues(flow, config, segment=False)\n    np.testing.assert_allclose(segmented["magnitude_variance"], 0.0)\n    np.testing.assert_allclose(unsegmented["magnitude_variance"], 1.0)\n    np.testing.assert_array_equal(unsegmented["segments"], 0)\n\n\ndef test_external_density_is_required_for_pressure_and_not_modified():\n    flow = np.full((3, 4, 2), 0.0)\n    flow[..., 0] = 2.0\n    density = np.arange(12, dtype=float).reshape(3, 4)\n    original = density.copy()\n    cues = compute_cues(flow, CueConfig(), density=density)\n    np.testing.assert_allclose(cues["pressure"], density * 2.0)\n    np.testing.assert_array_equal(density, original)\n\n\ndef test_segmentation_is_repeatable_with_bounded_training_sample():\n    flow = np.random.default_rng(4).normal(size=(20, 21, 2))\n    config = CueConfig(kmeans_samples=16, clusters=4, seed=7)\n    first = compute_cues(flow, config)\n    second = compute_cues(flow, config)\n    for name in first:\n        np.testing.assert_array_equal(first[name], second[name])\n    assert set(np.unique(first["segments"])) <= {-1, 0, 1, 2, 3}\n\n\ndef test_no_motion_has_zero_maps_and_background_labels():\n    result = compute_cues(np.zeros((2, 2, 2)), CueConfig())\n    assert not result["mask"].any()\n    np.testing.assert_array_equal(result["segments"], -1)\n    for name in ("magnitude", "compression", "direction_variance", "magnitude_variance"):\n        np.testing.assert_array_equal(result[name], 0)\n\n\n@pytest.mark.parametrize("flow", [np.zeros((2, 2)), np.zeros((1, 2, 2)), np.full((2, 2, 2), np.nan)])\ndef test_invalid_flow_is_rejected(flow):\n    with pytest.raises(ValueError):\n        compute_cues(flow, CueConfig())\n\n\n@pytest.mark.parametrize("density", [np.ones((3, 3)), -np.ones((2, 2)), np.full((2, 2), np.inf)])\ndef test_invalid_density_is_rejected(density):\n    with pytest.raises(ValueError):\n        compute_cues(np.ones((2, 2, 2)), CueConfig(), density=density)\n\n\n@pytest.mark.parametrize("kwargs", [{"clusters": 0}, {"clusters": True}, {"kmeans_samples": 0}, {"seed": -1}, {"min_motion": -1}, {"min_motion": np.nan}])\ndef test_invalid_configuration_is_rejected(kwargs):\n    with pytest.raises(ValueError):\n        CueConfig(**kwargs)\n\n\ndef test_top_fraction_aggregation_uses_full_area_unless_mask_supplied():\n    values = np.array([[0.0, 0.0], [0.0, 8.0]])\n    original = values.copy()\n    assert aggregate_map(values, top_fraction=0.5) == 4.0\n    assert aggregate_map(values, top_fraction=1.0) == 2.0\n    assert aggregate_map(values, top_fraction=0.5, mask=values > 0) == 8.0\n    assert aggregate_map(values, mask=np.zeros((2, 2), dtype=bool)) == 0.0\n    np.testing.assert_array_equal(values, original)\n\n\n@pytest.mark.parametrize("fraction", [0, -0.1, 1.1, np.nan])\ndef test_invalid_aggregation_fraction_is_rejected(fraction):\n    with pytest.raises(ValueError):\n        aggregate_map(np.ones((2, 2)), fraction)\n\n\ndef test_bad_aggregation_map_and_mask_are_rejected():\n    with pytest.raises(ValueError):\n        aggregate_map(np.array([[np.inf]]))\n    with pytest.raises(ValueError):\n        aggregate_map(-np.ones((2, 2)))\n    with pytest.raises(ValueError):\n        aggregate_map(np.ones((2, 2)), mask=np.ones((2, 2)))\n'}, 'tests/test_raftcrowd_io.py': {'sha256': 'e8b99fe026bf45deaf02e0631f110a9662c852b0653fb42f6a38a5114034262d', 'text': '"""Manifest/protocol admission checks using tiny synthetic files, without decoding."""\n\nimport csv\nimport json\n\nimport numpy as np\nimport pytest\n\nfrom raftcrowd.io import DEFAULT_CONFIG, load_config, load_manifest\n\n\n@pytest.fixture\ndef manifest_data(tmp_path):\n    rows = []\n    for split in ["train", "val", "test"]:\n        frames = tmp_path / f"{split}_frames"\n        frames.mkdir()\n        # Dummy bytes intentionally cannot be decoded: this tests admission only.\n        for name in ["000.png", "001.png", "002.png"]:\n            (frames / name).write_bytes(b"synthetic admission fixture")\n        labels = tmp_path / f"{split}_labels.npy"\n        np.save(labels, [0, 0, 0] if split == "train" else [0, 0, 1])\n        rows.append({\n            "id": f"video_{split}",\n            "split": split,\n            "group": f"source_{split}",\n            "frames": frames.name,\n            "labels": labels.name,\n            "density": "",\n        })\n    return tmp_path, rows\n\n\ndef write_manifest(tmp_path, rows, columns=None):\n    path = tmp_path / "manifest.csv"\n    with path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=columns or ["id", "split", "group", "frames", "labels", "density"])\n        writer.writeheader()\n        writer.writerows(rows)\n    return path\n\n\ndef test_defaults_and_partial_configuration(tmp_path):\n    defaults = load_config()\n    assert defaults == DEFAULT_CONFIG\n    defaults["clusters"] = 999\n    assert load_config()["clusters"] != 999\n    path = tmp_path / "config.json"\n    path.write_text(json.dumps({"seed": 17, "device": "cpu"}))\n    result = load_config(path)\n    assert result["seed"] == 17\n    assert result["device"] == "cpu"\n    assert result["height"] == DEFAULT_CONFIG["height"]\n\n\n@pytest.mark.parametrize(\n    "changes",\n    [\n        {"unknown": 1},\n        {"seed": True},\n        {"seed": -1},\n        {"height": 127},\n        {"height": 360.0},\n        {"updates": 0},\n        {"clusters": False},\n        {"kmeans_samples": 0},\n        {"normalization_max_samples": 0},\n        {"smooth_sigma": float("nan")},\n        {"min_motion": float("inf")},\n        {"top_fraction": True},\n        {"top_fraction": 0},\n        {"normalization_quantile": 1.1},\n        {"density_mode": "motion_proxy"},\n        {"weights": "DEFAULT"},\n        {"device": "mps"},\n        {"mixed_precision": 1},\n        {"device": "cpu", "mixed_precision": True},\n        {"density_mode": []},\n        {"weights": {}},\n        {"device": []},\n    ],\n)\ndef test_invalid_configuration_is_rejected(tmp_path, changes):\n    path = tmp_path / "config.json"\n    path.write_text(json.dumps(changes))\n    with pytest.raises(ValueError):\n        load_config(path)\n\n\n@pytest.mark.parametrize("value", [[], None, True, "config"])\ndef test_config_requires_object(tmp_path, value):\n    path = tmp_path / "config.json"\n    path.write_text(json.dumps(value))\n    with pytest.raises(ValueError):\n        load_config(path)\n\n\ndef test_manifest_resolves_paths_sorts_frames_and_loads_labels(manifest_data):\n    root, rows = manifest_data\n    result = load_manifest(write_manifest(root, rows), load_config())\n    assert [record["split"] for record in result] == ["train", "val", "test"]\n    assert [frame.name for frame in result[0]["frames"]] == ["000.png", "001.png", "002.png"]\n    assert all(frame.is_absolute() for record in result for frame in record["frames"])\n    np.testing.assert_array_equal(result[2]["labels"], [0, 0, 1])\n\n\ndef test_train_labels_are_optional_and_assumed_normal(manifest_data):\n    root, rows = manifest_data\n    rows[0]["labels"] = ""\n    result = load_manifest(write_manifest(root, rows), load_config())\n    assert result[0]["label_path"] is None\n    np.testing.assert_array_equal(result[0]["labels"], [0, 0, 0])\n\n\n@pytest.mark.parametrize("split_index", [1, 2])\ndef test_validation_and_test_labels_are_required(manifest_data, split_index):\n    root, rows = manifest_data\n    rows[split_index]["labels"] = ""\n    with pytest.raises(ValueError, match="need per-frame"):\n        load_manifest(write_manifest(root, rows), load_config())\n\n\ndef test_source_group_cannot_cross_splits(manifest_data):\n    root, rows = manifest_data\n    rows[1]["group"] = rows[0]["group"]\n    with pytest.raises(ValueError, match="crosses splits"):\n        load_manifest(write_manifest(root, rows), load_config())\n\n\ndef test_duplicate_resolved_frame_directories_rejected(manifest_data):\n    root, rows = manifest_data\n    rows[1]["frames"] = "./train_frames/../train_frames"\n    with pytest.raises(ValueError, match="reused"):\n        load_manifest(write_manifest(root, rows), load_config())\n\n\ndef test_anomalous_training_labels_rejected(manifest_data):\n    root, rows = manifest_data\n    np.save(root / rows[0]["labels"], [0, 1, 0])\n    with pytest.raises(ValueError, match="normal-only"):\n        load_manifest(write_manifest(root, rows), load_config())\n\n\n@pytest.mark.parametrize("labels", [[0, 1], [[0, 0, 1]], [0, 2, 1], [0, np.nan, 1]])\ndef test_invalid_label_shape_or_values_rejected(manifest_data, labels):\n    root, rows = manifest_data\n    np.save(root / rows[1]["labels"], labels)\n    with pytest.raises(ValueError, match="binary vector"):\n        load_manifest(write_manifest(root, rows), load_config())\n\n\ndef test_density_paths_rejected_when_density_disabled(manifest_data):\n    root, rows = manifest_data\n    rows[0]["density"] = "density_train"\n    with pytest.raises(ValueError, match="density_mode=none"):\n        load_manifest(write_manifest(root, rows), load_config())\n\n\ndef test_external_density_requires_all_video_directories(manifest_data):\n    root, rows = manifest_data\n    config = load_config()\n    config["density_mode"] = "external"\n    with pytest.raises(ValueError, match="every video"):\n        load_manifest(write_manifest(root, rows), config)\n    for row in rows:\n        row["density"] = f"density_{row[\'split\']}"\n        (root / row["density"]).mkdir()\n    result = load_manifest(write_manifest(root, rows), config)\n    assert all(record["density"].is_dir() for record in result)\n\n\n@pytest.mark.parametrize("video_id", ["../escape", "unsafe/name", "", "-prefix", "a b"])\ndef test_unsafe_video_ids_rejected(manifest_data, video_id):\n    root, rows = manifest_data\n    rows[0]["id"] = video_id\n    with pytest.raises(ValueError, match="safe names"):\n        load_manifest(write_manifest(root, rows), load_config())\n\n\ndef test_duplicate_ids_rejected(manifest_data):\n    root, rows = manifest_data\n    rows[1]["id"] = rows[0]["id"]\n    with pytest.raises(ValueError, match="safe names"):\n        load_manifest(write_manifest(root, rows), load_config())\n\n\ndef test_all_three_splits_required(manifest_data):\n    root, rows = manifest_data\n    with pytest.raises(ValueError, match="include train, val and test"):\n        load_manifest(write_manifest(root, rows[:2]), load_config())\n\n\ndef test_duplicate_image_stems_rejected(manifest_data):\n    root, rows = manifest_data\n    (root / rows[0]["frames"] / "000.jpg").write_bytes(b"duplicate stem")\n    with pytest.raises(ValueError, match="unique stems"):\n        load_manifest(write_manifest(root, rows), load_config())\n\n\ndef test_image_suffix_directories_do_not_count_as_frames(manifest_data):\n    root, rows = manifest_data\n    rows[0]["labels"] = ""\n    frames = root / rows[0]["frames"]\n    (frames / "001.png").unlink()\n    (frames / "002.png").unlink()\n    (frames / "001.png").mkdir()\n    with pytest.raises(ValueError, match=">=2 images"):\n        load_manifest(write_manifest(root, rows), load_config())\n\n\ndef test_duplicate_csv_headers_rejected(manifest_data):\n    root, rows = manifest_data\n    path = write_manifest(root, rows, ["id", "split", "group", "frames", "labels", "density", "id"])\n    with pytest.raises(ValueError, match="CSV columns"):\n        load_manifest(path, load_config())\n'}, 'tests/test_raftcrowd_metrics.py': {'sha256': '0668bbb36f660287bae37a931333d76137abf149b7441434dcc377d33c3e732c', 'text': '"""Synthetic analytical checks, not experimental evidence for the paper."""\n\nimport numpy as np\nimport pytest\n\nfrom raftcrowd.metrics import (\n    eer,\n    evaluate,\n    fixed_metrics,\n    roc_auc,\n    roc_curve,\n    select_threshold,\n)\n\n\ndef test_known_auc_and_curve():\n    y = [0, 0, 1, 1]\n    s = [0.1, 0.4, 0.35, 0.8]\n    fpr, tpr, thresholds = roc_curve(y, s)\n    np.testing.assert_allclose(fpr, [0, 0, 0.5, 0.5, 1])\n    np.testing.assert_allclose(tpr, [0, 0.5, 0.5, 1, 1])\n    np.testing.assert_allclose(thresholds[1:], [0.8, 0.4, 0.35, 0.1])\n    assert np.isposinf(thresholds[0])\n    assert roc_auc(y, s) == pytest.approx(0.75)\n    assert eer(y, s) == pytest.approx(0.5)\n\n\n@pytest.mark.parametrize("order", [[0, 1, 2, 3], [3, 1, 0, 2], [2, 0, 3, 1]])\ndef test_score_ties_are_order_invariant(order):\n    y = np.array([0, 1, 0, 1])[order]\n    s = np.array([0.2, 0.2, 0.8, 0.8])[order]\n    assert roc_auc(y, s) == pytest.approx(0.5)\n    assert eer(y, s) == pytest.approx(0.5)\n    assert len(roc_curve(y, s)[0]) == 3\n\n\n@pytest.mark.parametrize(\n    "scores,expected_auc,expected_eer",\n    [([0, 1], 1, 0), ([1, 0], 0, 1), ([0.5, 0.5], 0.5, 0.5)],\n)\ndef test_extreme_and_constant_predictions(scores, expected_auc, expected_eer):\n    assert roc_auc([0, 1], scores) == pytest.approx(expected_auc)\n    assert eer([0, 1], scores) == pytest.approx(expected_eer)\n\n\ndef test_eer_interpolates_between_curve_points():\n    # Descending labels 1,0,1,1 gives a crossing between (FPR=0,TPR=1/3)\n    # and (FPR=1,TPR=1/3), so EER is 2/3.\n    assert eer([1, 0, 1, 1], [4, 3, 2, 1]) == pytest.approx(2 / 3)\n\n\ndef test_fixed_metrics_threshold_is_inclusive_and_counts_are_exact():\n    result = fixed_metrics([0, 1, 0, 1], [0.1, 0.5, 0.5, 0.9], 0.5)\n    assert result == {\n        "precision": 2 / 3,\n        "recall": 1,\n        "f1": 0.8,\n        "tp": 2,\n        "fp": 1,\n        "tn": 1,\n        "fn": 0,\n    }\n\n\ndef test_zero_denominators_are_defined():\n    result = fixed_metrics([0, 0], [0.1, 0.2], 1)\n    assert result["precision"] == result["recall"] == result["f1"] == 0\n    assert result["tn"] == 2\n\n\ndef test_validation_threshold_selection_and_high_tie_break():\n    # Descending labels 1,0,0,1: F1=2/3 after the first and fourth rows.\n    y = [1, 0, 0, 1]\n    s = [0.9, 0.8, 0.7, 0.6]\n    assert select_threshold(y, s) == 0.9\n    assert select_threshold([0, 0, 1, 1], [0.1, 0.4, 0.35, 0.8]) == 0.35\n\n\ndef test_evaluate_does_not_optimize_supplied_threshold():\n    result = evaluate([0, 1], [0, 1], 2)\n    assert result["roc_auc"] == 1\n    assert result["f1"] == 0\n    assert result["fn"] == 1\n\n\n@pytest.mark.parametrize(\n    "labels,scores",\n    [\n        ([], []),\n        ([0, 1], [1]),\n        ([[0, 1]], [[0.1, 0.2]]),\n        ([0, 2], [0.1, 0.2]),\n        ([0, np.nan], [0.1, 0.2]),\n        ([0, 1], [0.1, np.inf]),\n        ([0, 1], [np.nan, 0.2]),\n        (["0", "1"], [0.1, 0.2]),\n        ([0, 1], [0.1 + 1j, 0.2]),\n    ],\n)\ndef test_invalid_arrays_rejected_by_all_entry_points(labels, scores):\n    for method in [roc_curve, roc_auc, eer, select_threshold]:\n        with pytest.raises(ValueError):\n            method(labels, scores)\n    with pytest.raises(ValueError):\n        fixed_metrics(labels, scores, 0.5)\n\n\n@pytest.mark.parametrize("labels", [[0, 0], [1, 1]])\ndef test_curve_and_threshold_selection_require_both_classes(labels):\n    for method in [roc_curve, roc_auc, eer, select_threshold]:\n        with pytest.raises(ValueError, match="both normal"):\n            method(labels, [0.1, 0.2])\n\n\n@pytest.mark.parametrize("threshold", [np.inf, np.nan, [0.5], "0.5", 0.5 + 1j])\ndef test_nonfinite_or_nonscalar_threshold_rejected(threshold):\n    with pytest.raises(ValueError, match="threshold"):\n        fixed_metrics([0, 1], [0.1, 0.2], threshold)\n'}, 'tests/test_raftcrowd_pipeline.py': {'sha256': '0c4def736fdb0552e3ed2d934ecd489d1d22a47212df9d7a20fe3f05fe457b8c', 'text': '"""Synthetic integration checks; these outputs are not scientific results."""\n\nimport csv\nimport json\n\nimport numpy as np\nimport pytest\n\nfrom raftcrowd.io import DEFAULT_CONFIG, sha256\nfrom raftcrowd.metrics import select_threshold\nfrom raftcrowd.pipeline import (\n    calibrate,\n    evaluate_cache,\n    load_cache,\n    prepare_cache,\n    score_map,\n)\nfrom raftcrowd.visualize import visualize\n\n\nclass SyntheticEstimator:\n    metadata = {"backend": "SYNTHETIC_TEST_ONLY", "purpose": "pipeline integration test"}\n\n    def __call__(self, first, second):\n        assert first.shape == second.shape == (128, 128, 3)\n        y, x = np.mgrid[:128, :128] / 127.0\n        amplitude = float(second[0, 0, 0]) / 255.0 + 0.5\n        flow = np.stack((amplitude * (2.0 - 0.5 * x), amplitude * (1.0 - 0.25 * y)), axis=-1)\n        return flow.astype(np.float32), 0.001\n\n\n@pytest.fixture\ndef synthetic_cache(tmp_path):\n    cv2 = pytest.importorskip("cv2")\n    config = dict(DEFAULT_CONFIG)\n    config.update(\n        device="cpu", height=128, width=128, smooth_sigma=0.0,\n        clusters=2, min_motion=0.0, kmeans_samples=32,\n        normalization_samples_per_frame=32, normalization_max_samples=64,\n    )\n    config_path = tmp_path / "config.json"\n    config_path.write_text(json.dumps(config), encoding="utf-8")\n    manifest_path = tmp_path / "manifest.csv"\n    with manifest_path.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=["id", "split", "group", "frames", "labels", "density"])\n        writer.writeheader()\n        for split_index, split in enumerate(["train", "val", "test"]):\n            directory = tmp_path / split\n            directory.mkdir()\n            for frame_index in range(3):\n                brightness = 10 + 30 * split_index + 10 * frame_index\n                frame = np.full((128, 128, 3), brightness, dtype=np.uint8)\n                assert cv2.imwrite(str(directory / f"frame_{frame_index:04d}.png"), frame)\n            label_path = tmp_path / f"{split}_labels.npy"\n            labels = [0, 0, 0] if split == "train" else [0, 0, 1]\n            np.save(label_path, np.array(labels, dtype=np.uint8), allow_pickle=False)\n            writer.writerow({\n                "id": f"{split}_video", "split": split, "group": f"{split}_source",\n                "frames": directory.name, "labels": label_path.name, "density": "",\n            })\n    cache = tmp_path / "cache"\n    meta = prepare_cache(manifest_path, config_path, cache, estimator=SyntheticEstimator())\n    return {"root": tmp_path, "cache": cache, "meta": meta,\n            "manifest": manifest_path, "config": config_path}\n\n\ndef test_extract_calibrate_evaluate_preserves_frozen_protocol(synthetic_cache):\n    setup = synthetic_cache\n    cache, root = setup["cache"], setup["root"]\n    assert setup["meta"]["estimator"]["backend"] == "SYNTHETIC_TEST_ONLY"\n    assert (cache / "complete.json").is_file()\n    assert [p["index"] for p in setup["meta"]["videos"][0]["pairs"]] == [1, 2]\n    model_path = root / "model.json"\n    model = calibrate(cache, model_path)\n    assert model["threshold_source"] == "validation_labels_only"\n    assert model["normalization"]["training_pairs"] == 2\n    assert set(model["thresholds"]) == set(model["methods"])\n    assert "pressure_only" not in model["methods"]\n    assert len(model["validation_predictions"]) == 2\n    assert all(row["video"] == "val_video" for row in model["validation_predictions"])\n    for method, threshold in model["thresholds"].items():\n        validation = model["validation_predictions"]\n        assert threshold == select_threshold(\n            [row["label"] for row in validation], [row[method] for row in validation]\n        )\n    report = evaluate_cache(cache, model_path, root / "evaluation")\n    assert report["aggregation"] == "pooled_scored_frames"\n    assert "first frame excluded" in report["frame_alignment"]\n    assert set(report["methods"]) == set(model["methods"])\n    for method, results in report["methods"].items():\n        pooled = results["pooled"]\n        assert results["threshold"] == model["thresholds"][method]\n        assert sum(pooled[key] for key in ["tp", "tn", "fp", "fn"]) == 2\n        assert 0 <= pooled["roc_auc"] <= 1\n        assert 0 <= pooled["eer"] <= 1\n        assert set(results["per_video"]) == {"test_video"}\n    evaluation = root / "evaluation"\n    with (evaluation / "frame_scores.csv").open(newline="", encoding="utf-8") as handle:\n        predictions = list(csv.DictReader(handle))\n    assert [row["index"] for row in predictions] == ["1", "2"]\n    assert [row["label"] for row in predictions] == ["0", "1"]\n    receipt = json.loads((evaluation / "complete.json").read_text(encoding="utf-8"))\n    assert receipt["metrics_sha256"] == sha256(evaluation / "metrics.json")\n    assert receipt["frame_scores_sha256"] == sha256(evaluation / "frame_scores.csv")\n\n\ndef test_calibration_does_not_consult_test_labels(synthetic_cache):\n    setup = synthetic_cache\n    original = calibrate(setup["cache"], setup["root"] / "original_model.json")\n    # Simulate a separately versioned input annotation set: both source and\n    # cached copies and their receipts are updated. Train/val inputs are fixed.\n    changed_labels = np.array([0, 1, 0], dtype=np.uint8)\n    source_labels = setup["root"] / "test_labels.npy"\n    cached_labels = setup["cache"] / "test_video" / "labels.npy"\n    np.save(source_labels, changed_labels, allow_pickle=False)\n    np.save(cached_labels, changed_labels, allow_pickle=False)\n    cache_json = setup["cache"] / "cache.json"\n    meta = json.loads(cache_json.read_text(encoding="utf-8"))\n    test_record = next(v for v in meta["videos"] if v["split"] == "test")\n    test_record["labels_sha256"] = sha256(cached_labels)\n    test_record["label_source_sha256"] = sha256(source_labels)\n    cache_json.write_text(json.dumps(meta), encoding="utf-8")\n    (setup["cache"] / "complete.json").write_text(\n        json.dumps({"cache_sha256": sha256(cache_json)}), encoding="utf-8"\n    )\n    changed = calibrate(setup["cache"], setup["root"] / "changed_model.json")\n    assert changed["cache_sha256"] != original["cache_sha256"]\n    for key in ["thresholds", "scales", "normalization", "validation_predictions"]:\n        assert changed[key] == original[key]\n\n\ndef test_corrupt_training_pair_is_rejected(synthetic_cache):\n    setup = synthetic_cache\n    file = setup["cache"] / "train_video" / "pair_000001.npz"\n    file.write_bytes(file.read_bytes() + b"corruption")\n    with pytest.raises(ValueError, match="Cached pair checksum mismatch"):\n        calibrate(setup["cache"], setup["root"] / "model.json")\n    assert not (setup["root"] / "model.json").exists()\n\n\ndef test_corrupt_test_pair_is_rejected_at_evaluation(synthetic_cache):\n    setup = synthetic_cache\n    model_path = setup["root"] / "model.json"\n    calibrate(setup["cache"], model_path)\n    file = setup["cache"] / "test_video" / "pair_000002.npz"\n    file.write_bytes(file.read_bytes() + b"corruption")\n    with pytest.raises(ValueError, match="Cached pair checksum mismatch"):\n        evaluate_cache(setup["cache"], model_path, setup["root"] / "evaluation")\n    assert not (setup["root"] / "evaluation" / "complete.json").exists()\n\n\ndef test_cache_metadata_receipt_is_checked(synthetic_cache):\n    setup = synthetic_cache\n    file = setup["cache"] / "cache.json"\n    file.write_bytes(file.read_bytes() + b" ")\n    with pytest.raises(ValueError, match="Cache metadata checksum mismatch"):\n        load_cache(setup["cache"])\n\n\ndef test_existing_scientific_outputs_are_not_overwritten(synthetic_cache):\n    setup = synthetic_cache\n    model_path = setup["root"] / "model.json"\n    calibrate(setup["cache"], model_path)\n    model_hash = sha256(model_path)\n    with pytest.raises(FileExistsError):\n        calibrate(setup["cache"], model_path)\n    assert sha256(model_path) == model_hash\n    output = setup["root"] / "evaluation"\n    evaluate_cache(setup["cache"], model_path, output)\n    report_hash = sha256(output / "metrics.json")\n    with pytest.raises(FileExistsError):\n        evaluate_cache(setup["cache"], model_path, output)\n    assert sha256(output / "metrics.json") == report_hash\n    with pytest.raises(FileExistsError):\n        prepare_cache(setup["manifest"], setup["config"], setup["cache"], estimator=SyntheticEstimator())\n\n\ndef test_score_map_uses_frozen_scales_and_cue_mean():\n    arrays = {\n        "compression": np.array([[0.0, 1.0], [4.0, 8.0]]),\n        "magnitude_variance": np.array([[1.0, 2.0], [3.0, 4.0]]),\n        "global_magnitude_variance": np.full((2, 2), 8.0),\n    }\n    scales = {"compression": 4.0, "magnitude_variance": 4.0,\n              "global_magnitude_variance": 16.0}\n    method = {"cues": ["compression", "magnitude_variance"], "segmented": True}\n    result = score_map(arrays, method, scales)\n    np.testing.assert_allclose(result, [[0.125, 0.375], [0.875, 1.0]])\n    global_method = dict(method, segmented=False)\n    global_result = score_map(arrays, global_method, scales)\n    np.testing.assert_allclose(global_result, [[0.25, 0.375], [0.75, 0.75]])\n\n\ndef test_visualization_writes_six_panel_png_exclusively(synthetic_cache):\n    cv2 = pytest.importorskip("cv2")\n    setup = synthetic_cache\n    model_path = setup["root"] / "model.json"\n    calibrate(setup["cache"], model_path)\n    output = setup["root"] / "six_panel.png"\n    assert visualize(setup["cache"], model_path, "test_video", 1, output) == output\n    decoded = cv2.imread(str(output), cv2.IMREAD_COLOR)\n    assert decoded is not None\n    assert decoded.shape == (3 * (128 + 32), 2 * 128, 3)\n    assert np.any(decoded > 0)\n    before = sha256(output)\n    with pytest.raises(FileExistsError):\n        visualize(setup["cache"], model_path, "test_video", 1, output)\n    assert sha256(output) == before\n\n\ndef test_visualization_rejects_unscored_first_frame(synthetic_cache):\n    setup = synthetic_cache\n    model_path = setup["root"] / "model.json"\n    calibrate(setup["cache"], model_path)\n    output = setup["root"] / "invalid_frame.png"\n    with pytest.raises(ValueError, match="scored second-frame index"):\n        visualize(setup["cache"], model_path, "test_video", 0, output)\n    assert not output.exists()\n\n\ndef test_visualization_rejects_changed_source_frame(synthetic_cache):\n    setup = synthetic_cache\n    model_path = setup["root"] / "model.json"\n    calibrate(setup["cache"], model_path)\n    source = setup["root"] / "test" / "frame_0000.png"\n    source.write_bytes(source.read_bytes() + b"changed source identity")\n    output = setup["root"] / "changed_source.png"\n    with pytest.raises(ValueError, match="Source image identity changed"):\n        visualize(setup["cache"], model_path, "test_video", 1, output)\n    assert not output.exists()\n\n\ndef _enable_external_density(setup):\n    config = json.loads(setup["config"].read_text(encoding="utf-8"))\n    config["density_mode"] = "external"\n    setup["config"].write_text(json.dumps(config), encoding="utf-8")\n    with setup["manifest"].open(newline="", encoding="utf-8") as handle:\n        reader = csv.DictReader(handle)\n        fieldnames, rows = reader.fieldnames, list(reader)\n    for row in rows:\n        directory = setup["root"] / f"{row[\'split\']}_density"\n        directory.mkdir()\n        # Only source grids for the two evaluated pairs are supplied. There is\n        # intentionally no density for the final frame, which has no outgoing flow.\n        np.save(directory / "frame_0000.npy", np.full((128, 128), 2.0), allow_pickle=False)\n        np.save(directory / "frame_0001.npy", np.full((128, 128), 7.0), allow_pickle=False)\n        row["density"] = directory.name\n    with setup["manifest"].open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fieldnames)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef test_external_density_uses_source_frame_grid_and_enables_pressure_variants(synthetic_cache):\n    setup = synthetic_cache\n    _enable_external_density(setup)\n    cache = setup["root"] / "external_density_cache"\n    meta = prepare_cache(setup["manifest"], setup["config"], cache, estimator=SyntheticEstimator())\n    for video in meta["videos"]:\n        for pair in video["pairs"]:\n            index = pair["index"]\n            expected_density = 2.0 if index == 1 else 7.0\n            source = setup["root"] / f"{video[\'split\']}_density" / f"frame_{index - 1:04d}.npy"\n            assert pair["source_density_sha256"] == sha256(source)\n            with np.load(cache / video["id"] / pair["file"], allow_pickle=False) as arrays:\n                magnitude = np.hypot(arrays["flow"][..., 0], arrays["flow"][..., 1])\n                np.testing.assert_allclose(arrays["pressure"], expected_density * magnitude, rtol=1e-6)\n    model_path = setup["root"] / "external_model.json"\n    model = calibrate(cache, model_path)\n    assert "pressure" in model["methods"]["full"]["cues"]\n    assert model["methods"]["pressure_only"]["cues"] == ["pressure"]\n    assert "pressure" not in model["methods"]["without_pressure"]["cues"]\n    assert model["scales"]["pressure"] > 0\n    report = evaluate_cache(cache, model_path, setup["root"] / "external_evaluation")\n    assert {"pressure_only", "without_pressure"} <= set(report["methods"])\n\n\ndef test_external_density_rejects_automatic_grid_resizing(synthetic_cache):\n    setup = synthetic_cache\n    _enable_external_density(setup)\n    density = setup["root"] / "train_density" / "frame_0000.npy"\n    np.save(density, np.ones((127, 128)), allow_pickle=False)\n    cache = setup["root"] / "bad_density_cache"\n    with pytest.raises(ValueError, match="aligned to the resized source-frame grid"):\n        prepare_cache(setup["manifest"], setup["config"], cache, estimator=SyntheticEstimator())\n    assert not (cache / "complete.json").exists()\n'}, 'scripts/raftcrowd.py': {'sha256': '21ce6733ebbc92da14c6c946e12d5e2dd6e9ee3a34c9aced29b2123fe304b57c', 'text': '"""Run from any working directory: python /path/to/scripts/raftcrowd.py --help."""\n\nfrom pathlib import Path\nimport sys\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))\n\nfrom raftcrowd.cli import main\n\nif __name__ == "__main__":\n    main()\n'}, 'configs/raftcrowd.json': {'sha256': 'b69fb75d023f62398da222776f8a6697242827e8b801c10de4d88817ae991487', 'text': '{\n  "seed": 7,\n  "height": 360,\n  "width": 640,\n  "weights": "C_T_V1",\n  "updates": 12,\n  "device": "cuda",\n  "mixed_precision": false,\n  "smooth_sigma": 1.0,\n  "clusters": 4,\n  "min_motion": 0.5,\n  "kmeans_iterations": 20,\n  "kmeans_samples": 4096,\n  "density_mode": "none",\n  "top_fraction": 0.01,\n  "normalization_quantile": 0.99,\n  "normalization_samples_per_frame": 128,\n  "normalization_max_samples": 50000\n}\n'}, 'configs/raftcrowd_manifest.example.csv': {'sha256': 'bc2d536e3e80244282a1d65467512e11ceb7328514a16096132a5b66edeac47e', 'text': 'id,split,group,frames,labels,density\ntrain001,train,source_train,../data/raftcrowd/train001,,\nval001,val,source_val,../data/raftcrowd/val001,../data/raftcrowd/val001_labels.npy,\ntest001,test,source_test,../data/raftcrowd/test001,../data/raftcrowd/test001_labels.npy,\n'}, 'requirements-raftcrowd.txt': {'sha256': '244b94dc7159829f28052f2714be3a13581326073a5e883280c3fe2ed2a0e740', 'text': '# Install torch and torchvision as a compatible pair for your CUDA runtime first.\n# The broad bounds below are compatibility declarations, not a frozen environment.\n# Each extraction records actual versions; freeze a tested environment before experiments.\nnumpy>=2.0,<3\nopencv-python-headless>=4.8,<6\ntorch>=2.2,<3\ntorchvision>=0.17,<1\n'}, 'RAFTCROWD_README.md': {'sha256': 'f582078154d104b6c683d58ea8a6387acccb966becc5e0effa3097ed8e721cfe', 'text': '# RAFT-CrowdFlow: code from the beginning\n\nThis is a new reference implementation based on the manuscript\'s architecture. It makes previously unspecified choices explicit. It does **not** reproduce the rejected paper\'s AUC values or validate crowd-crush prediction. The original notebook/PDFs and the separate LiteCrowdRelNet repository are unchanged.\n\n## Start in Colab\n\nOpen `output/jupyter-notebook/RAFTCrowdFlow_from_scratch.ipynb` in Colab and select a GPU runtime. The notebook embeds this package, configuration, instructions and synthetic tests with source checksums, so cloning the repository is optional. Follow the cells to install dependencies, check the implementation, supply frame data/labels, extract flow, calibrate, evaluate and display the six-panel image.\n\nThe notebook is generated by `python scripts/build_raftcrowd_notebook.py`; edit reusable sources and rebuild, rather than editing embedded source by hand. Its source pins preserve the exact generated implementation. New code means a new extraction cache.\n\n## Local or Linux GPU setup\n\nUse a clean Python environment and install a compatible PyTorch/Torchvision pair for your GPU. Then install the declared dependencies:\n\n```sh\npython -m pip install -r requirements-raftcrowd.txt\npython scripts/raftcrowd.py --help\n```\n\nThe dependency ranges are not a tested lockfile. Actual versions, backend, hardware, precision, command, configuration, input hashes and source hashes are recorded during extraction. Pin the actual working environment before a research run. The default 360 x 640 RAFT-large inference can require substantial GPU memory; CPU is supported explicitly but slow. No full RAFT or GPU run was performed in the implementation environment.\n\n## Data layout\n\nSupply a CSV manifest with exactly these columns. Paths are relative to the manifest file, or absolute. Each frame directory contains chronologically sorted, zero-padded PNG/JPG filenames. Use distinct `group` values across splits for the actual independent source units (camera/sequence/scene as appropriate); software cannot infer undisclosed related sources.\n\n```csv\nid,split,group,frames,labels,density\ntrain001,train,source_train,train001/frames,,\nval001,val,source_val,val001/frames,val001/labels.npy,\ntest001,test,source_test,test001/frames,test001/labels.npy,\n```\n\nLabels are one-dimensional binary `.npy` arrays of length N: 0 normal, 1 anomalous, aligned to input frame filenames. Derive these from the dataset\'s actual annotations; never create arbitrary labels for research results. Omitted train labels declare normal-only data; validation and test labels are required. Each of the pooled validation and pooled test sets needs both classes after the first frame is excluded. Multiple datasets should have separate manifests/caches, not pooled dataset scores.\n\nTo decode a local video into frames:\n\n```sh\npython scripts/raftcrowd.py video-frames --video path/to/video.mp4 --out data/clip_frames\n```\n\nThis extracts every decoded frame and records diagnostic decoder timestamps, not a validated native-PTS alignment. For standard benchmarks, use the official frame files and exact official annotation alignment whenever available.\n\n## Run the complete workflow\n\nEdit the example manifest/configuration for your actual files. Defaults select CUDA and the explicit pretrained `C_T_V1` RAFT-large weights. The weights download on the first model construction. Set `device` to `cpu` if needed.\n\n```sh\npython scripts/raftcrowd.py extract --manifest configs/raftcrowd_manifest.example.csv --config configs/raftcrowd.json --out results/raftcrowd/cache\npython scripts/raftcrowd.py calibrate --cache results/raftcrowd/cache --out results/raftcrowd/model.json\npython scripts/raftcrowd.py evaluate --cache results/raftcrowd/cache --model results/raftcrowd/model.json --out results/raftcrowd/evaluation\npython scripts/raftcrowd.py visualize --cache results/raftcrowd/cache --model results/raftcrowd/model.json --video-id test001 --frame-index 1 --out results/raftcrowd/panel.png\n```\n\nOutputs are exclusive-create: choose a new directory/path for another run. Interrupted extraction leaves a partial cache without a completion receipt; it cannot be calibrated. This version does not implement cache resumption. All flow/cue maps are cached per pair, so budget storage before a full benchmark. At 360 x 640 each float32 scalar map is about 0.92 MB before compression; this implementation stores several maps plus two-channel flow and labels per pair. Start with a small **separate** pilot manifest before scheduling full extraction.\n\nThe model JSON stores training-derived normalization scales, validation-chosen thresholds for every method and validation predictions. Test evaluation writes `metrics.json`, `frame_scores.csv` and a checksum receipt, including pooled and per-video metrics. AUC/EER are fractions, not percentages. Single-class per-video AUC/EER are null with a reason; the pooled test set requires both classes. First frames have no score: pair `(t,t+1)` is evaluated against label `t+1`.\n\n## What is implemented\n\n- One frozen RAFT field per observed frame pair, RGB preprocessing, padding removed by cropping.\n- Gaussian smoothing of numeric flow; deterministic motion-vector clustering.\n- Nonnegative compression, circular directional variance, regional magnitude variance.\n- Training-only cue normalization and equal-weight CFI fusion; top-fraction frame scoring.\n- Flow-magnitude-only baseline, individual cues, leave-one-cue-out controls, and a no-segmentation control with its own training-derived variability scales.\n- Validation-only threshold selection; fixed-threshold test F1, pooled ROC-AUC/EER and per-frame output records.\n- Six panels from the same cached flow: source RGB, flow color, CFI, magnitude, compression and motion groups.\n\nNo anomaly network is trained: RAFT is pretrained and frozen; "training" fits cue scales. There is no optimizer or epoch schedule to invent.\n\n## Optional pedestrian density\n\nThe default is a **motion-only CFI**. To add the manuscript\'s fourth term, set `density_mode` to `external` and provide a density directory for every manifest row. Supply `source_frame_stem.npy` for each pair\'s first frame. Each map must be finite, nonnegative and already aligned to the configured resized H x W grid. It is multiplied by flow magnitude. Automatic resizing is rejected because count-density transforms require a declared area correction. Record the density estimator, checkpoint, training data, units and transformations separately.\n\nNo Gaussian-blurred flow is presented as pedestrian density. Until real density maps are supplied, pressure experiments are unavailable; the default `full` name means **all active motion cues**, not the paper\'s unverified four-term configuration. No temporal mask warping is implemented in this first version. K-means can group spatially disconnected motion, and the moving-pixel gate can miss near-static crowd compression. Image divergence is not physical pressure.\n\n## Validation and method notes\n\nRun the synthetic CPU suite without installing PyTorch:\n\n```sh\n# Linux / Colab\nPYTHONPATH=src python -m pytest -q tests/test_raftcrowd_*.py\n```\n\n```powershell\n# PowerShell\n$env:PYTHONPATH = \'src\'\n& \'.venv/Scripts/python.exe\' -m pytest -q tests/test_raftcrowd_*.py\n```\n\nThese tests check cue mathematics, metrics, split/label admission and the extraction-to-evaluation workflow using an explicitly synthetic injected estimator. They do not establish RAFT accuracy, real-data performance, novelty or deployment speed.\n\nMethod choices and limitations: `docs/raftcrowd-method.md`. Prospective experiment protocol: `plan/raftcrowd-experiment-protocol.md`. Output schema: `tables/raftcrowd-table-schema.md`. Figure provenance: `figures/raftcrowd-data-manifest.md`.\n\nOfficial API references: [Torchvision RAFT Large](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.optical_flow.raft_large.html) and [optical-flow tutorial](https://docs.pytorch.org/vision/stable/auto_examples/others/plot_optical_flow.html). These document the pretrained model and RGB preprocessing, not this method\'s experimental effectiveness.\n'}, 'docs/raftcrowd-method.md': {'sha256': 'd317976cedadd181b1ff9f2e7ea994e928d8fd502af38bcd2501ec67a53dc7d7', 'text': "# RAFT-CrowdFlow starting implementation\n\nDate: 2026-11-18. This is a new, explicitly specified implementation inspired by the rejected manuscript. It does not recover the missing original evaluation code, reproduce its tables, establish novelty, or validate crowd-crush prediction. The SDD LiteCrowdRelNet project is separate and unchanged.\n\n## Pipeline\n\n```text\nordered RGB frames I_t, I_(t+1)\n    -> one frozen RAFT flow field F_t\n    -> magnitude and moving-pixel mask\n    -> deterministic motion segmentation\n    -> compression + regional directional/magnitude variability\n       + optional external pedestrian-density x magnitude\n    -> training-fitted cue scaling\n    -> fixed-weight CFI map\n    -> top-fraction spatial aggregation\n    -> anomaly score available at frame t+1\n```\n\nThere is no learned anomaly network or optimizer in this version. RAFT uses pretrained weights without updating them. Training data fit cue normalization; validation data select a fixed operating threshold for each declared method. Test data only evaluate those frozen choices.\n\n## Explicit implementation choices\n\n| Component | Starting specification | Limit or departure from the manuscript |\n| --- | --- | --- |\n| Flow estimator | Torchvision `raft_large`, explicit `Raft_Large_Weights.C_T_V1` | FlyingChairs/FlyingThings training lineage; not a byte-identical copy of the uploaded notebook's `raft-things.pth` file |\n| Inputs | Consecutive RGB frames, resized to 360 x 640 by default, batch size 1 | Displacements are pixels in this resized image per pair, not original-image pixels, metres or velocity per second |\n| Geometry | Replicate-pad images to a multiple of eight, run RAFT, crop the output flow back to the resized image | Crop, rather than resize, removes padding; input height and width must satisfy RAFT's supported minimum |\n| RAFT updates | 12 by default; floating point 32 by default | These settings do not reproduce the paper's unverified timing; optional CUDA mixed precision is an explicit configuration choice |\n| Flow smoothing | Optional Gaussian smoothing of numeric u/v fields, default sigma 1 | Smoothing acts on flow, not on a color visualization |\n| Moving mask | Magnitude greater than 0.5 pixels per pair by default | This threshold is a proposed setting, not a recovered experimental value |\n| Segmentation | At most four deterministic k-means groups using moving-pixel u/v vectors; bounded fitting sample and fixed seed | No semantic person segmentation, no temporal mask propagation/warping in version 1 |\n| Compression | `max(0, -(du/dx + dv/dy))` | Finite differences use the resized image grid; this is a visual motion cue, not a calibrated physical pressure measurement |\n| Directional variability | Per-region circular variance `1 - hypot(mean(cos(theta)), mean(sin(theta)))` | Explicitly replaces naive linear angle variance so directions near -pi and pi remain close |\n| Magnitude variability | Per-region population variance of flow magnitude | Broadcast to that region's moving pixels; zero outside the moving mask |\n| Density coupling | Optional `D_t * magnitude(F_t)` from externally supplied, nonnegative pedestrian-density maps | No density estimator is invented or inferred from flow. Default pressure cue is disabled. Blurred flow magnitude is not pedestrian density |\n| Scaling | Training-only sampled 99th-percentile upper scale per nonnegative cue; zero lower bound; clamp to [0,1] | No per-test-video min-max normalization and no test-fitted scaling |\n| Fusion | Equal weights for active CFI cues, renormalized for each declared ablation | These weights are proposed and fixed; not the manuscript's unknown weights |\n| Frame score | Mean of the largest 1% of map pixels, over the whole image | Uses all spatial pixels for a consistent denominator; no best-test-score aggregation search |\n\nMap names used by the implementation include `magnitude`, `compression`, `direction_variance`, `magnitude_variance` and optional `pressure`. Segmentation labels and the moving mask support interpretation but are not extra CFI terms. Empty moving regions produce zero variability rather than missing scores.\n\nThe initial configuration records seed 7, at most 20 k-means updates on at most 4,096 fitting vectors, and normalization sampling of 128 values per frame capped at 50,000 per cue. These are implementation defaults, not values recovered from the rejected paper. The cache also stores the single-region control so ablation scoring does not rerun RAFT under different inputs.\n\nNormalization samples only moving pixels from normal training pairs. A bounded random-priority reservoir is deterministic for a fixed manifest traversal order. An empty or effectively zero-range training cue receives an explicitly recorded scale of 1 to avoid division by zero; inspect `zero_scale_fallback_cues` before interpreting its contribution. Single-region variability maps have their own training-derived scales. Scaling is fixed before validation chooses thresholds.\n\n## Coordinates, timing and density\n\nThe flow from frames t and t+1 lives on source-frame t coordinates. Any external density map must refer to those same source-frame coordinates and the configured resized resolution. Name its `.npy` file using the source frame's stem. A density array must be a finite, nonnegative H x W NumPy array; the pipeline must reject a missing or mismatched density map when pressure is enabled. Density provenance must name its estimator, checkpoint, training data, calibration, coordinate transform and units. It must not be a flow-magnitude image relabeled as a count map.\n\nThe resulting pair score becomes available only when frame t+1 has arrived. Evaluation therefore aligns it with `labels[t+1]`; frame zero has no pair score and is excluded. This convention measures anomaly discrimination, not advance prediction. No warning lead time is established without separate annotated event-onset analysis.\n\nThe resized coordinate system and camera perspective limit comparisons of motion magnitude across scenes. Neither a CFI heatmap nor negative image divergence independently proves crowd compression, crowd density, a dangerous interaction or causal risk. Heatmaps require separate spatial annotations before localization accuracy can be claimed.\n\n## Controls and claim-to-experiment traceability\n\nAll methods must use the same frame pairs, RAFT flow and input cohort. The magnitude baseline must not receive different normalization, labels or threshold selection privileges.\n\n| Question | Required comparison/evidence | Permitted conclusion after successful execution |\n| --- | --- | --- |\n| Does CFI add information beyond speed? | Full active CFI versus flow-magnitude-only baseline | Dataset-specific anomaly-score comparison, not safety effectiveness |\n| Which cue contributes? | Each cue alone and each leave-one-cue-out variant | Conditional contribution under the declared fixed scoring protocol |\n| Does segmentation contribute? | Full CFI versus a single-moving-region control | Effect of this specific segmentation construction |\n| Does pedestrian density contribute? | Same cohort with external density and pressure removed; inspect density provenance | Effect of that supplied estimator's density coupling |\n| Are operating-point results valid? | Validation-chosen threshold frozen before test, saved per-frame predictions | Held-out precision, recall and F1 for that fixed threshold |\n| Are reported AUC values reproducible? | Archived configuration, manifest, input identities, code and predictions | Regenerated values from the new implementation only |\n| Is early crush warning supported? | Separate crowd-crush/onset annotations and prospective lead-time/false-alarm evaluation | No such conclusion from the current anomaly benchmarks alone |\n| Is embedded real time supported? | Actual target-device benchmark with synchronized timing, warm-up and latency distribution | No such conclusion from video playback FPS or Colab metadata |\n\n## Primary implementation references\n\n- [Torchvision RAFT Large API and weight definitions](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.optical_flow.raft_large.html).\n- [Official Torchvision optical-flow example](https://docs.pytorch.org/vision/stable/auto_examples/others/plot_optical_flow.html), including RGB preprocessing, pretrained transforms and output convention.\n\nThese sources document the RAFT API. They do not establish novelty, experimental results, crowd-safety validity or equivalence to the rejected paper.\n"}, 'plan/raftcrowd-experiment-protocol.md': {'sha256': '81c5237dbc5aa37053ccd6ca0cdd39ebc323182bc33d61ae2de97e64692024e6', 'text': "# RAFT-CrowdFlow prospective experiment protocol\n\nDate: 2026-11-18. Phase: method implementation and experiment planning. This protocol concerns the new RAFT video-anomaly implementation, not the SDD trajectory project documented in `experiment-protocol.md`. No benchmark performance is recorded here.\n\n## Scope and preregistered choices\n\nQuestion: does a specified combination of compression and regional motion variability improve frame-level anomaly discrimination over a matched flow-magnitude score? Pedestrian-density coupling is an optional, separately declared study requiring external density maps. Crush prediction, prevention, physical pressure and spatial risk localization are not validated tasks in this protocol.\n\nThe initial method specification is in `../docs/raftcrowd-method.md`. Freeze configuration, cue list, equal fusion weights, moving threshold, clustering seed, training normalization, aggregation rule, datasets and split manifest before test evaluation. A change after inspecting test results creates an exploratory variant; it does not recover an untouched holdout. The paper's historical test exposure must be disclosed if the same videos are reused.\n\n## Data manifest and exclusions\n\nUse a CSV with columns `id,split,group,frames,labels,density`. Each row identifies a video represented as an ordered directory of frames. Valid split values are `train`, `val` and `test`.\n\n| Field | Meaning |\n| --- | --- |\n| `id` | Unique stable video identifier, safe for output naming |\n| `split` | Role fixed before scoring; all three roles required for the full workflow |\n| `group` | Leakage-prevention grouping such as recording, camera or scene, chosen according to the study |\n| `frames` | Directory containing sorted PNG/JPG frames with zero-padded names and no duplicate frame stems |\n| `labels` | NumPy vector of N binary frame labels in that exact order; required for validation/test; training may omit labels only when all frames are declared normal |\n| `density` | Optional directory of aligned source-frame pedestrian-density arrays; required when pressure is enabled |\n\nDeclare how groups were constructed and verify that train/validation/test groups are disjoint. Do not silently mix normal and abnormal excerpts from the same original recording across splits. Duplicate frame directories must be rejected. Group checks cannot prove independence if authors assign misleading or incomplete groups; inspect original recording provenance as well.\n\nRecord dataset release/version and access terms, official split correspondence or any departures, frame extraction cadence, image ordering, label source, exclusions and input hashes. Frame decoding/extraction must preserve every intended frame in order; video codec metadata alone is not proof of alignment. Exclude frame zero from pair-score metrics in every method. Stop on malformed labels, fewer than two frames, absent required inputs or density-coordinate mismatches rather than dropping rows.\n\n## Fixed preprocessing and method\n\n- Default resized shape: H=360, W=640; RGB uint8 input; consecutive frame pairs; numeric flow cropped after padding.\n- Frozen RAFT Large weights: `Raft_Large_Weights.C_T_V1`; 12 updates; batch 1; fp32 default. Any mixed-precision variant has its own configuration and result identity.\n- Proposed defaults: Gaussian flow smoothing sigma=1; moving magnitude threshold=0.5 resized pixels/pair; at most four motion groups; deterministic seed 7.\n- Cues: nonnegative compression, circular directional variance and regional magnitude variance. Pressure is absent unless valid external pedestrian density is explicitly enabled.\n- Fit per-cue scaling from deterministic bounded samples of training maps only, using a zero lower bound and 99th-percentile upper scale. Freeze scales before validation/test scoring; preserve the sampling procedure and counts.\n- Full active CFI uses fixed equal cue weights. The frame score is the mean of the largest 1% of pixels over the whole map.\n- RAFT weights do not train; optimizer, learning rate, epochs, early stopping and model-selection checkpoints are not applicable. Calibration outputs and threshold locks replace trainable anomaly-model artifacts.\n\nThe starting JSON configuration additionally records `kmeans_iterations=20`, `kmeans_samples=4096`, `normalization_samples_per_frame=128` and `normalization_max_samples=50000`. The density mode is explicitly `none` or `external`; a missing density input must not be silently replaced by motion magnitude.\n\n## Executable workflow and artifact layout\n\nFrom the coordination repository root, after providing actual frames and editing the example manifest:\n\n```bash\npython scripts/raftcrowd.py extract --manifest configs/raftcrowd_manifest.example.csv --config configs/raftcrowd.json --out results/raftcrowd/cache\npython scripts/raftcrowd.py calibrate --cache results/raftcrowd/cache --out results/raftcrowd/model.json\npython scripts/raftcrowd.py evaluate --cache results/raftcrowd/cache --model results/raftcrowd/model.json --out results/raftcrowd/evaluation\npython scripts/raftcrowd.py visualize --cache results/raftcrowd/cache --model results/raftcrowd/model.json --video-id test001 --frame-index 1 --out results/raftcrowd/panel.png\n```\n\nThe example manifest contains example paths, not a downloaded dataset. GPU/Colab dependencies and pretrained weights are required for real extraction; the coordinator's native Windows environment has not validated GPU execution. Change device/resolution settings deliberately and retain the resulting configuration identity.\n\nExtraction writes per-pair numeric NPZ shards under `cache/<video_id>/pair_000001.npz`, with the index naming the second/availability frame. This keeps extraction streaming rather than loading the entire benchmark into RAM. It does not make dense maps small: at 360 x 640, one float32 map is about 0.92 MB before compression, and storing many cues/controls over many frames can consume substantial disk space. Estimate storage with a small pilot before full extraction; compression ratios depend on the actual data. Keep source data, caches and result outputs local and out of Git.\n\nThe calibration JSON freezes cue scales and validation-selected thresholds. Evaluation and visualization must load those frozen values. Existing outputs must not be overwritten; use new paths for a new run. Preserve the cache receipt and input identity alongside the model/evaluation artifacts. An interrupted or partial cache is not complete scientific evidence.\n\n## Comparator fairness\n\nRun all configurations on the exact same admitted video/frame-pair cohort:\n\n1. Flow-magnitude-only score.\n2. Each active CFI cue alone.\n3. Full active CFI.\n4. Full CFI with each active cue removed in turn.\n5. Full CFI with all moving pixels assigned to one region, retaining the other rules.\n\nPressure comparisons require the same external density coverage in every method. Changing the cohort to accommodate density is a separate declared analysis, not a replacement for the original comparison. Train the normalization for the no-segmentation control using its own training-derived cue distribution; keep its data and scaling policy matched. Give each method its own validation-selected operational threshold without allowing test selection.\n\nThe method is not an RGB-only trainable anomaly baseline, and this implementation does not fabricate that comparator. Published comparator values require verified primary sources and compatible protocols. In particular, the reviewer-requested HF2-VAD comparison needs a separate primary-source/protocol audit before reporting a numerical comparison.\n\n## Calibration and threshold freeze\n\nTraining labels, when supplied, must establish normal-only calibration data. Validation must contain both classes for F1-based threshold selection. Select the threshold maximizing validation F1, using the highest threshold to break ties. Save threshold values, validation input identity, calibration identity and selection rule in a new output location. A test run may load this frozen artifact; it may not alter it using test scores or labels.\n\nMaximum F1 obtained on test labels must never be described as deployment performance. No per-test-video min-max normalization, cue-weight optimization, cluster tuning or aggregation selection is permitted. If the model is revised after test inspection, label the following results exploratory and obtain genuinely unseen data for confirmatory claims.\n\n## Metrics and uncertainty\n\nReport micro-pooled frame ROC-AUC and EER using all retained pair scores and binary labels; also report fixed-threshold precision, recall, F1 and confusion counts. Preserve per-video metrics and sample counts. ROC-AUC and EER are undefined for a single-class video; show null with an explanation rather than a manufactured zero. A pooled metric is not the mean of per-video metrics; label the aggregation explicitly.\n\nNo repeated-run variability, confidence interval or significance claim is supported until the appropriate runs or resampling analysis have actually been performed. If confidence intervals are later added, account for temporal correlation by resampling videos or independent groups rather than treating adjacent frames as independent samples. Report failed cases and negative ablations without suppressing them.\n\n## Runtime and reproducibility record\n\nFor each run archive the exact command, config JSON, code revision and dirty state or source digest, manifest and split identities, input/label/density hashes, RAFT weight identity, random seed, software versions, Python interpreter, OS, device and output location. Preserve numeric prediction arrays and generated table provenance. Never overwrite historical experiment output directories.\n\nIf benchmarking efficiency, record device name, power mode, input shape, batch, precision, RAFT update count, warm-up duration/count, timed pair count, CUDA synchronization, peak memory and mean/median/P95 latency. State whether decoding, resizing, RAFT, segmentation, density inference, CFI, visualization and file writing are included. Report FPS separately from encoded video playback speed. No Jetson speed number from the rejected paper is imported as a verified result.\n\n## Completion gates\n\n| Gate | Required evidence | Current evidence status |\n| --- | --- | --- |\n| Method specification | Source agrees with formulas, defaults and deliberate departures | New specification; coordinator must cross-check final code |\n| Unit/integration correctness | Executed transformations, metric, split and pipeline tests | Actual test outcome belongs in progress/handoff, not assumed here |\n| Real RAFT execution | Supported environment, real frames, weights and numeric output | NOT_DONE in this protocol |\n| Reproducible benchmark results | Real manifests, labels, configs, predictions and regenerated tables | NOT_DONE |\n| Safety/early-warning claim | Appropriate event annotations, lead-time and false-alarm study | NOT_DONE |\n\nSynthetic fixtures verify implementation behavior. They are not anomaly-benchmark evidence, and any generated demonstration data must be marked `mock_` / `PLANNING DATA - NOT EXPERIMENTAL EVIDENCE` when placed in a result-facing artifact.\n"}, 'tables/raftcrowd-table-schema.md': {'sha256': '5d43700de49263981bce02d1619a335b79721186feb29bf34a79b4a472dd8fac', 'text': "# RAFT-CrowdFlow result table schema\n\nDate: 2026-11-18. This file defines future outputs; it contains no measured results. Never copy the rejected manuscript's 98.9/91.4/73.4 values into new result tables as defaults.\n\n## Comparison and ablation rows\n\nOne row per dataset, manifest identity, method/configuration and evaluation aggregation:\n\n| Column | Meaning |\n| --- | --- |\n| `dataset` / `dataset_version` | Explicit release or author-defined dataset identity |\n| `manifest_hash` | Frozen video/split/label manifest identity |\n| `code_revision` / `source_digest` | Actual executable source identity; record dirty state |\n| `config_hash` | Flow, cues, normalization, fusion and aggregation identity |\n| `method` | Magnitude, cue-alone, full CFI, cue removed or no segmentation |\n| `density_source` | Disabled, or estimator/checkpoint/coordinate provenance |\n| `seed` | Actual deterministic seed; no invented repeated-run variability |\n| `split` / `aggregation` | Test/validation and micro-pooled/per-video explicitly |\n| `video_count` / `frame_pair_count` | Retained evaluation cohort, excluding every first frame |\n| `roc_auc` | ROC-AUC as fraction [0,1]; multiply by 100 only in a clearly labeled display |\n| `eer` | EER fraction [0,1]; display percent only when labeled |\n| `threshold` | Actual frozen numeric operational threshold |\n| `threshold_source` | Validation F1, selection rule and artifact identity |\n| `precision`, `recall`, `f1` | Fixed-threshold metrics, fractions [0,1] |\n| `tp`, `fp`, `tn`, `fn` | Integer confusion counts |\n| `prediction_artifact` | Exact per-frame score file and hash |\n| `evidence_status` | Executed/verified or explicitly pending; never imply a planned run occurred |\n\nFor per-video single-class rows, ROC-AUC/EER must be null and the reason stated. Keep those videos in pooled analysis if their inputs are valid. Missing density or malformed labels must not silently change one method's cohort.\n\nThe full-CFI row in the comparison table and full-CFI row in the ablation table must reference the same predictions when claiming the same dataset/protocol. Generate both displays from one reviewed result source; reconcile any differences before manuscript editing.\n\n## Operating-threshold table\n\nIf reporting several alarm thresholds, name the dataset, split, unit of evaluation, score definition and numeric threshold in each row. Distinguish a validation threshold sweep from fixed-threshold held-out performance. Repeated values across displays are acceptable only when they intentionally describe the same predictions and operating point.\n\nFrame-label precision/recall/F1 must not be labeled risk-localization performance. Spatial localization requires a separate table with spatial ground truth, region/pixel matching rules and a validated evaluation implementation.\n\n## Efficiency table\n\nInclude hardware, OS/software, power mode, input dimensions, batch, precision, RAFT updates, timed/warm-up counts, synchronized timing boundary, included components, mean/median/P95 milliseconds, throughput pairs/s, peak memory and timing-log artifact. Do not infer processing speed from a video writer's FPS setting. Results on Colab cannot be renamed Jetson measurements.\n\n## Consistency checks before publication\n\nVerify precision/recall/F1 against confusion counts, units across every display, cohort counts across methods, threshold source independence, and table-to-prediction hashes. Report rounding only in presentation; retain full numeric precision in artifacts. No AUC difference alone establishes statistical significance.\n"}, 'figures/raftcrowd-data-manifest.md': {'sha256': '9b1876595ef41b24a72584fd0e47ffd432a92236e5cac3fe4725d4609e0938dc', 'text': '# RAFT-CrowdFlow figure provenance requirements\n\nDate: 2026-11-18. No new scientific figure is claimed in this manifest. These requirements apply to figures produced by the new implementation, not retrospectively verified figures from the rejected PDF.\n\nFor each figure retain source video/frame names and hashes, manifest/config/source identities, original and resized dimensions, pair indices, RAFT weights, numeric flow and cue arrays, calibration/threshold artifacts, plotting settings, output path and caption. Use one flow tensor for all panels belonging to a pair. Record the time at which the displayed score becomes available.\n\n| Panel/output | Numerical source | Required labeling |\n| --- | --- | --- |\n| RGB source | Source frame I_t | Video, frame index, resolution and any resize/crop |\n| Flow color | Same numeric F_t used by every derived panel | Color direction convention; magnitude display scaling |\n| CFI map | Normalized active cues with declared weights | Actual CFI; cue set, fixed scale and available-at-frame t+1 convention |\n| Magnitude | `sqrt(u^2+v^2)` | Resized pixels per pair, not metres/s or density |\n| Divergence/compression | Numeric signed divergence or nonnegative compression | Name the chosen quantity; fixed zero/scale and finite-difference convention |\n| Density | External aligned pedestrian-density map only | Estimator and units; no density panel when density is absent |\n| Segmentation | Deterministic motion-region labels | Motion clusters, not person segmentation or annotated risk zones |\n\nDo not substitute flow arrows for CFI or blurred flow magnitude for pedestrian density. A colorized map is a visualization, not the numeric scoring input. Use consistent fixed scales when making between-frame intensity comparisons; if a visualization uses independent display normalization, disclose it and do not infer changes in calibrated score from color alone.\n\nQuantitative plots and tables must read reviewed numeric artifacts, not manually entered manuscript values. Synthetic examples must be clearly labeled `mock_` / `PLANNING DATA - NOT EXPERIMENTAL EVIDENCE`. Qualitative frames may illustrate behavior but do not validate early warning, physical pressure or spatial localization without task-matched annotations and quantitative evaluation.\n'}}

for name, item in FILES.items():
    target = ROOT / name
    data = item['text'].encode('utf-8')
    assert hashlib.sha256(data).hexdigest() == item['sha256'], name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        if target.read_bytes() != data:
            raise RuntimeError(f'Existing source differs: {target}; choose a fresh ROOT')
    else:
        target.write_bytes(data)
sys.path.insert(0, str(ROOT / 'src'))
os.environ['PYTHONPATH'] = str(ROOT / 'src')
print(f'Checked {len(FILES)} source files in {ROOT}')


## Environment and synthetic checks

The following cell installs the declared dependencies in your runtime. Colab normally provides a compatible PyTorch/Torchvision GPU pair; check the displayed versions. These broad dependency bounds are not a tested lockfile. Synthetic checks do not run RAFT or establish dataset performance.


In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements-raftcrowd.txt'), 'pytest'])
import torch, torchvision, numpy as np
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('CUDA:', torch.cuda.is_available())
subprocess.check_call([sys.executable, '-m', 'pytest', '-q', str(ROOT / 'tests')], cwd=ROOT)


## Supply benchmark frames and actual labels

Prepare a ZIP with `manifest.csv` at its root and frame/label folders. See the embedded `RAFTCROWD_README.md` for exact formats. A row example is `val001,val,camera_val,val001/frames,val001/labels.npy,`.

Use separate train/validation/test source groups. Train must be normal-only. Labels are binary `.npy` vectors of length N, one for each chronologically named input frame. Both classes must occur among scored frames in the pooled validation and pooled test sets. Do not invent labels. Run separate manifests/caches for each dataset.

The cell below uploads and extracts a local ZIP. Alternatively skip it and set `MANIFEST` to a file you have placed in this runtime or mounted storage. No dataset is downloaded by the code. For video inputs, first use `python scripts/raftcrowd.py video-frames --video ... --out ...`, then align true annotations explicitly.


In [ ]:
# Colab upload: alternatively skip and set MANIFEST in the next cell.
from google.colab import files
import stat, zipfile
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Upload one dataset ZIP with manifest.csv at its root')
DATA = ROOT / 'input_data'
DATA.mkdir(exist_ok=False)
with zipfile.ZipFile(zip_names[0]) as archive:
    for member in archive.infolist():
        target = (DATA / member.filename).resolve()
        if not target.is_relative_to(DATA.resolve()) or stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError('Archive has an unsafe path or link')
    archive.extractall(DATA)
MANIFEST = DATA / 'manifest.csv'
assert MANIFEST.is_file(), 'manifest.csv must be at the ZIP root'


## Freeze configuration before extraction

Defaults: RAFT-large C_T_V1, 360 x 640, 12 updates, FP32, four motion groups, fixed equal weights, training-only 99th-percentile cue scaling, top-1% mean frame score. The output flow is cropped after padding. No temporal region warping is implemented.

Optional density: set `density_mode='external'` and supply a density folder in **every** manifest row. Maps must be independently produced pedestrian-density arrays, already aligned to the resized source frame, named with its stem. Record their estimator/checkpoint/units separately. The default has no pressure term.

Caching several numeric maps for every frame pair can use substantial storage. Start with a separate small pilot manifest to check inputs and resources. This implementation has no extraction resume; keep failed outputs for inspection and use a new run directory.


In [ ]:
from datetime import datetime, timezone
from raftcrowd.io import load_config, load_manifest

# For mounted/pre-existing data, set MANIFEST = Path('/path/to/manifest.csv') here.
config = load_config(ROOT / 'configs/raftcrowd.json')
config['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
# Make deliberate prospective changes here, before running this experiment.
RUN = ROOT / 'results' / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
RUN.mkdir(parents=True, exist_ok=False)
CONFIG = RUN / 'config.json'
CONFIG.write_text(json.dumps(config, indent=2), encoding='utf-8')
records = load_manifest(MANIFEST, load_config(CONFIG))
print([(r['id'], r['split'], len(r['frames'])) for r in records])
print('Run:', RUN, 'Device:', config['device'])


## Extract frozen RAFT flow and numeric cues

This is the costly step. Pretrained weights download on first use. Every panel, baseline and CFI cue uses the same cached field; a model is never silently switched between panels. Original frames and labels are hashed. No benchmark metrics or thresholds are selected here.


In [ ]:
def run_cli(*args):
    subprocess.check_call([sys.executable, str(ROOT / 'scripts/raftcrowd.py'), *map(str, args)], cwd=ROOT)

CACHE = RUN / 'cache'
run_cli('extract', '--manifest', MANIFEST, '--config', CONFIG, '--out', CACHE)


## Calibrate using training and validation only

Normal training frames fit cue scales. Validation labels choose each method's F1 threshold. The model JSON saves these fixed choices and validation predictions. A no-segmentation control has separately fitted training variability scales. Test labels are not used for selection.


In [ ]:
MODEL = RUN / 'model.json'
run_cli('calibrate', '--cache', CACHE, '--out', MODEL)


## Evaluate the held-out test set

The output includes full active CFI, magnitude-only, individual cue, leave-one-cue-out and no-segmentation controls on identical pairs. It reports pooled frame ROC-AUC, interpolated EER and fixed-threshold precision/recall/F1, plus per-video metrics and per-frame scores. Values are fractions, not percentages. Pair `(t,t+1)` is evaluated at label `t+1`; frame zero is excluded.

Do not tune this configuration after reading test results and describe the same test set as untouched. These anomaly metrics do not establish early warning, prevention or spatial localization of crowd crushes.


In [ ]:
EVALUATION = RUN / 'evaluation'
run_cli('evaluate', '--cache', CACHE, '--model', MODEL, '--out', EVALUATION)
report = json.loads((EVALUATION / 'metrics.json').read_text())
for name, item in report['methods'].items():
    print(name, item['pooled'])


## Six panels from one flow field

Panels show source RGB, flow color, actual CFI score, magnitude, compression and motion groups. Colors represent normalized motion cues, not verified safety risk. The preview requires the unchanged original frame files as well as the cache.


In [ ]:
from IPython.display import Image, display
TEST_VIDEO = next(r['id'] for r in records if r['split'] == 'test')
PANEL = RUN / 'six_panel.png'
run_cli('visualize', '--cache', CACHE, '--model', MODEL, '--video-id', TEST_VIDEO,
        '--frame-index', 1, '--out', PANEL)
display(Image(filename=str(PANEL), width=1000))


## Preserve evidence

Keep the original manifest, frame/annotation identities, density provenance if used, source pins, cache receipts, model JSON, prediction CSV and metric JSON. Extraction timing is diagnostic, has no warm-up and excludes parts of the pipeline; it is not a Jetson or end-to-end deployment benchmark.

The source package includes method choices, a prospective protocol and table/figure provenance schemas. Historical scores from the rejected paper remain unverified. No new claim of novelty, improvement or safety effectiveness follows from the presence of this code.
